In [2]:
import numpy as np
import pandas as pd
import torch
import os
import random
import pyarrow as pa
from scipy import sparse
from pathlib import Path

In [3]:
# =========================
# Step 1 · 环境与依赖初始化（PyTorch 多 GPU 基座）
# =========================

# ---- 标准库 ----
import os
import re
import math
import json
import time
import random
from pathlib import Path
from typing import Optional, Tuple, List

# ---- 科学计算 / 数据 IO ----
import numpy as np
import pandas as pd

# Parquet 要求：使用 fastparquet
# pip install fastparquet
# （若需要更高性能也可选 pyarrow，但本项目统一 fastparquet）
PARQUET_ENGINE = "fastparquet"

# 稀疏矩阵（用于与已有管线兼容的三元组落盘；随后将逐步转换为 PyTorch 稀疏）
from scipy import sparse

# ---- PyTorch 栈 ----
import torch
import torch.nn as nn
import torch.nn.functional as F

# 可选：分布式 / 多 GPU。Notebook 推荐 DataParallel；脚本/集群可用 DDP（torchrun）
import torch.distributed as dist

# 提升 matmul 的 float32 精度/吞吐（PyTorch 2.0+）
try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

# cudnn benchmark 有益于卷积型工作负载；对本项目影响不大，但开启通常无害
torch.backends.cudnn.benchmark = True

# ---- 近邻搜索（后续 Step 7+ 用）----
# 推荐 GPU 版：pip install faiss-gpu==1.7.4
# CPU 版：pip install faiss-cpu==1.7.4
try:
    import faiss  # noqa
    _FAISS_AVAILABLE = True
except Exception:
    _FAISS_AVAILABLE = False

# ---- 通用路径 ----
TMP_DIR = Path("./tmp")  # 统一的中间件目录（你要求）
TMP_DIR.mkdir(exist_ok=True)

# ---- 随机种子 ----
def set_seed(seed: int = 2025):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(2025)

# ---- GPU/分布式环境工具 ----
def gpu_summary():
    n = torch.cuda.device_count()
    if n == 0:
        print("[GPU] No CUDA device found. Running on CPU.")
        return
    print(f"[GPU] CUDA devices: {n}")
    for i in range(n):
        prop = torch.cuda.get_device_properties(i)
        print(f"  - GPU{i}: {prop.name}, {prop.total_memory/1024**3:.1f} GB, SMs={prop.multi_processor_count}")

def get_device() -> torch.device:
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")

def use_all_gpus_for_module(module: nn.Module) -> nn.Module:
    """
    Notebook/单进程场景：用 DataParallel 覆盖所有 GPU。
    如使用 torchrun/Slurm 多进程 DDP，请在外部初始化后改用 DDP 包装。
    """
    if torch.cuda.is_available() and torch.cuda.device_count() > 1:
        return nn.DataParallel(module)  # 简便多GPU
    return module

def maybe_init_ddp(backend: str = "nccl"):
    """
    仅当你用 torchrun 启动时才会生效；Notebook 默认不做 DDP 初始化。
    export MASTER_ADDR=... MASTER_PORT=... WORLD_SIZE=... RANK=...
    torchrun --nproc_per_node=NUM_GPUS your_script.py
    """
    if "RANK" in os.environ and "WORLD_SIZE" in os.environ:
        if not dist.is_initialized():
            dist.init_process_group(backend=backend)
            local_rank = int(os.environ.get("LOCAL_RANK", 0))
            torch.cuda.set_device(local_rank)
            print(f"[DDP] Initialized. rank={dist.get_rank()}/{dist.get_world_size()}, local_rank={local_rank}")
    else:
        # Notebook 情况：不启 DDP
        pass

gpu_summary()
print(f"[Env] TMP_DIR = {TMP_DIR.resolve()}")

# ---- Parquet / 稀疏 CSR 三元组 读写工具（与历史管线兼容） ----
def save_parquet_df(df: pd.DataFrame, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(path, engine=PARQUET_ENGINE, index=False)

def load_parquet_df(path: Path) -> pd.DataFrame:
    return pd.read_parquet(path, engine=PARQUET_ENGINE)

def save_csr_as_triplets(mat: sparse.csr_matrix, path: Path, dtype=np.float32):
    coo = mat.tocoo()
    pdf = pd.DataFrame({
        "row": coo.row.astype(np.int32),
        "col": coo.col.astype(np.int32),
        "val": coo.data.astype(dtype),
    })
    save_parquet_df(pdf, path)

def load_csr_from_triplets(path: Path, shape: Tuple[int, int], dtype=np.float32) -> sparse.csr_matrix:
    pdf = load_parquet_df(path)
    coo = sparse.coo_matrix((pdf["val"].astype(dtype), (pdf["row"], pdf["col"])), shape=shape, dtype=dtype)
    return coo.tocsr()

# ---- Torch 稀疏工具（后续步骤会逐步改为 torch.sparse 格式进行 GPU 计算）----
def csr_to_torch_coo(mat: sparse.csr_matrix, device: torch.device = None, dtype: torch.dtype = torch.float32):
    mat = mat.tocoo()
    indices = torch.tensor(np.vstack([mat.row, mat.col]), dtype=torch.long, device=device)
    values  = torch.tensor(mat.data, dtype=dtype, device=device)
    shape   = torch.Size(mat.shape)
    return torch.sparse_coo_tensor(indices, values, shape, device=device, dtype=dtype).coalesce()

def torch_row_l2_normalize(x: torch.Tensor, eps: float = 1e-12) -> torch.Tensor:
    """
    对 dense 或稀疏的行做 L2 归一（对稀疏COO：只缩放 values）。
    这里先提供 dense 版本；后续在具体步骤对稀疏做定制归一。
    """
    if x.is_sparse:
        # 稀疏归一在后续针对性实现；此处仅放占位说明
        raise NotImplementedError("Sparse L2 normalize will be implemented in the graph-specific steps.")
    norm = torch.linalg.norm(x, dim=1, keepdim=True).clamp_min(eps)
    return x / norm

print("[Step 1] PyTorch 多 GPU 环境初始化完成。后续步骤将沿用以上工具与路径。")


[GPU] CUDA devices: 2
  - GPU0: NVIDIA RTX 6000 Ada Generation, 47.5 GB, SMs=142
  - GPU1: NVIDIA RTX 6000 Ada Generation, 47.5 GB, SMs=142
[Env] TMP_DIR = /home/koyo/workspace/recsys/tmp
[Step 1] PyTorch 多 GPU 环境初始化完成。后续步骤将沿用以上工具与路径。


In [4]:
# =========================
# Step 2 · Params: I/O + 预处理（保持结构，落盘到 TMP_DIR）
# =========================

from pathlib import Path

# 输入数据路径
DATA_CSV_PATH = Path("./data/metadata_merged.csv")

# 构建 text_all 的原始列（按需要可增/减）
TEXT_COLS = ["Title", "Subtitle", "Description", "Slug"]

# 标签列（CSV中的原始列名）
TAG_COL = "Tags"

# 主键列（数据集ID）
ID_COL = "Id"

# 可能存在的时间列（择优使用）
CREATED_COL_CANDIDATES     = ["CreationDate_dt", "CreationDate"]
LAST_ACTIVE_COL_CANDIDATES = ["LastActivityDate_dt", "LastActivityDate"]

# 文本标准化（轻量）
TEXT_LOWER      = True
TEXT_STRIP_HTML = True   # 粗略移除 <...> 标记
TEXT_DROP_URL   = True   # 粗略移除 http(s) URL

# 保存文件名
DOC_CLEAN_PATH = TMP_DIR / "doc_clean.parquet"
INDEX_MAP_PATH = TMP_DIR / "index_map.parquet"


In [6]:
# =========================
# Step 2 · Execute: 读取CSV→清洗→合列text_all→标签解析→落盘
# =========================

import re
import pandas as pd
import numpy as np

# 1) 读取
df = pd.read_csv(DATA_CSV_PATH, low_memory=False)
print(f"[INFO] 读入: {DATA_CSV_PATH}  shape={df.shape}")

# 2) 选择时间列（有 *_dt 优先）
def _pick_first_exist(cols, candidates):
    for c in candidates:
        if c in cols:
            return c
    return None

created_col = _pick_first_exist(df.columns, CREATED_COL_CANDIDATES)
lastact_col = _pick_first_exist(df.columns, LAST_ACTIVE_COL_CANDIDATES)

# 3) 规范 Id
if ID_COL not in df.columns:
    raise ValueError(f"未找到主键列: {ID_COL}")
# 若有重复 Id，保留第一条
df = df.drop_duplicates(subset=[ID_COL], keep="first").reset_index(drop=True)
df[ID_COL] = pd.to_numeric(df[ID_COL], errors="coerce").astype("Int64")

# 4) 合列 text_all
def _safe_str(x):
    if pd.isna(x): return ""
    return str(x)

text_pieces = []
for col in TEXT_COLS:
    if col in df.columns:
        text_pieces.append(df[col].map(_safe_str))
if text_pieces:
    text_all = text_pieces[0]
    for p in text_pieces[1:]:
        text_all = text_all + " " + p
else:
    text_all = pd.Series([""] * len(df), dtype=str)

# 轻量清洗：去HTML尖括号、URL、小写化
if TEXT_STRIP_HTML:
    text_all = text_all.str.replace(r"<[^>]+>", " ", regex=True)
if TEXT_DROP_URL:
    text_all = text_all.str.replace(r"https?://\S+", " ", regex=True)
if TEXT_LOWER:
    text_all = text_all.str.lower()

# 5) 解析标签 → 存储为逗号分隔字符串（兼容 fastparquet）
def _parse_tags(x: str):
    if pd.isna(x) or not isinstance(x, str): return ""
    # 拆分逗号，去空格，小写；去除空tag
    tags = [t.strip().lower() for t in x.split(",")]
    tags = [t for t in tags if t]
    return ",".join(tags)

if TAG_COL in df.columns:
    tag_str = df[TAG_COL].map(_parse_tags)
else:
    tag_str = pd.Series([""] * len(df), dtype=str)

# 6) 规范化时间
def _to_dt_safe(x):
    try:
        return pd.to_datetime(x, errors="coerce")
    except Exception:
        return pd.NaT

created_at = _to_dt_safe(df[created_col]) if created_col else pd.Series([pd.NaT]*len(df))
last_active_at = _to_dt_safe(df[lastact_col]) if lastact_col else pd.Series([pd.NaT]*len(df))

# 7) 组装 doc_df（保持与后续步骤对齐）
doc_df = pd.DataFrame({
    "doc_idx": np.arange(len(df), dtype=np.int64),
    "Id":      df[ID_COL].astype("Int64"),
    "text_all": text_all.fillna(""),
    "tag_str":  tag_str.fillna(""),
    "created_at": created_at,
    "last_active_at": last_active_at,
})

# 8) 基础统计
n_docs = len(doc_df)
docs_with_tags = int((doc_df["tag_str"].str.len() > 0).sum())
avg_words = float(doc_df["text_all"].str.split().map(len).replace(0, np.nan).mean())
print(f"[INFO] 文档数: {n_docs:,}")
print(f"[INFO] 含标签文档数/占比: {docs_with_tags:,} / {n_docs:,} ({docs_with_tags/n_docs:.1%})")
print(f"[INFO] 平均文本词数(粗略): {avg_words:.2f}")

# 9) 落盘（Parquet / fastparquet）
save_parquet_df(doc_df, DOC_CLEAN_PATH)
index_map = doc_df[["doc_idx","Id"]].copy()
save_parquet_df(index_map, INDEX_MAP_PATH)
print(f"[OK] 已保存：\n  - {DOC_CLEAN_PATH}\n  - {INDEX_MAP_PATH}")


[INFO] 读入: data/metadata_merged.csv  shape=(521735, 31)
[INFO] 文档数: 521,735
[INFO] 含标签文档数/占比: 214,603 / 521,735 (41.1%)
[INFO] 平均文本词数(粗略): 41.41
[OK] 已保存：
  - tmp/doc_clean.parquet
  - tmp/index_map.parquet


In [7]:
# Quick sanity checks for Step 2 outputs
from pathlib import Path
import pandas as pd
import numpy as np

TMP_DIR = Path("./tmp")

doc = pd.read_parquet(TMP_DIR / "doc_clean.parquet", engine="fastparquet")
idx = pd.read_parquet(TMP_DIR / "index_map.parquet", engine="fastparquet")

print(f"[CHECK] doc_clean shape={doc.shape}, index_map shape={idx.shape}")
print("[CHECK] columns:", list(doc.columns))

# 1) 前3行预览（截断文本方便查看）
preview = doc[["doc_idx","Id","tag_str","created_at","last_active_at","text_all"]].head(3).copy()
preview["text_all"] = preview["text_all"].str.slice(0, 120) + "..."
print("\n[PREVIEW]\n", preview.to_string(index=False))

# 2) 唯一性与取值范围
ok_unique = doc["doc_idx"].is_unique
ok_range  = (doc["doc_idx"].min()==0) and (doc["doc_idx"].max()==len(doc)-1)
dup_id    = doc["Id"].duplicated().sum()
print(f"\n[CHECK] doc_idx unique={ok_unique}, contiguous={ok_range}, duplicated Ids={dup_id}")

# 3) 关键列缺失情况
na_stats = doc[["Id","text_all","tag_str","created_at","last_active_at"]].isna().sum()
print("\n[NA counts]\n", na_stats)

# 4) 文本长度分布（词数）
tok_len = doc["text_all"].fillna("").str.split().map(len)
print("\n[TEXT LEN] mean/median/min/max = ",
      f"{tok_len.mean():.2f}/{tok_len.median()}/{tok_len.min()}/{tok_len.max()}")

# 5) 标签覆盖与Top-15标签（若有tag_str）
has_tag = (doc["tag_str"].str.len() > 0)
print(f"\n[TAGS] docs_with_tags={has_tag.sum():,} / {len(doc):,} ({has_tag.mean():.1%})")
if has_tag.any():
    tags_exploded = doc.loc[has_tag, "tag_str"].str.split(",").explode().str.strip()
    top_tags = tags_exploded.value_counts().head(15)
    print("\n[TAGS] top-15\n", top_tags.to_string())

# 6) index_map与doc对齐性
merged = idx.merge(doc[["doc_idx","Id"]], on=["doc_idx","Id"], how="outer", indicator=True)
mismatch = (merged["_merge"] != "both").sum()
print(f"\n[CHECK] index_map alignment mismatches = {mismatch}")

print("\n[OK] Step 2 sanity checks completed.")


[CHECK] doc_clean shape=(521735, 6), index_map shape=(521735, 2)
[CHECK] columns: ['doc_idx', 'Id', 'text_all', 'tag_str', 'created_at', 'last_active_at']

[PREVIEW]
  doc_idx  Id                                      tag_str          created_at last_active_at                                                                                                                    text_all
       0   6 computer science,demographics,social science 2015-07-18 00:51:12     2018-02-05 2013 american community survey find insights in the 2013 american community survey the [american community survey](  is ...
       1   7      internet,linguistics,online communities 2015-08-04 23:59:00     2018-02-06 may 2015 reddit comments get personal with a dataset of comments from may 2015 recently reddit released [an enormous dat...
       2   8              atmospheric science,environment 2015-08-18 21:53:00     2018-01-31 ocean ship logbooks (1750-1850) explore changing climatology with data from early shippin

In [10]:
# =========================
# Step 3 · Params: Tag 视图（GPU）—— D–T Binary / TF-IDF / PPMI
# =========================

from pathlib import Path

# 读写目录（统一）
TMP_DIR = Path("./tmp")

# 词表过滤
MIN_DF_TAG = 10        # 至少在多少文档中出现的标签才保留
MAX_TAGS   = None      # 可设置为 int 做顶部截断；None 表示不截断

# TF-IDF 参数（对 Tag 语境，tf=1，idf 为平滑IDF）
TFIDF_IDF_SMOOTH = True        # True: idf = log((N+1)/(df+1)) + 1; False: idf = log(N/df)
ROW_NORM_TFIDF   = "l2"         # {"l2","l1",None}

# PPMI 参数（对非零项计算 pmi = log( total / (row_deg * col_deg) ) ，再截断为正）
PPMI_EPS         = 1e-12
ROW_NORM_PPMI    = "l2"         # {"l2","l1",None}

# 多 GPU 切分（对非零条目切块并行）
USE_ALL_GPUS = True             # 尽量使用所有 GPU
CHUNKS_PER_GPU = 1              # 每块大小由 nnz / (num_gpus*chunks_per_gpu) 决定


In [15]:
# =========================
# Step 4 · Params: Text 视图（GPU）—— D–W BM25
# =========================

from pathlib import Path

# 统一TMP路径（沿用你的写法）
TMP_DIR = Path("./tmp")
TMP_DIR.mkdir(exist_ok=True)

# —— 词表筛选 —— 
MIN_DF_W        = 200       # 词在多少文档中出现才保留（与之前保持一致）
MAX_DF_RATIO_W  = 0.50      # 丢弃出现在>50%文档中的“过常见词”
KEEP_TOP_W      = None      # 也可额外设定保留Top-N高df词，None表示不额外截断

# —— 分词 ——（轻量可控；与原先一致的英文token思路）
TOKEN_PATTERN   = r"[a-z0-9_]+"
MIN_TOKEN_LEN   = 2
TO_LOWER        = True

# 可选停用词（保持简洁；你有更全词表也可替换）
STOPWORDS = {
    "the","and","to","of","in","for","on","with","a","is","this","that",
    "it","as","from","by","an","or","be","are","at","we","you","your",
}

# —— BM25 参数 ——（与 IR 常用默认一致）
BM25_K1 = 1.5
BM25_B  = 0.75

# 行归一（便于后续图扩散）
ROW_NORM_BM25 = "l2"   # {"l2","l1",None}

# —— 性能参数 —— 
CHUNK_DOCS   = 50_000  # CSV→文本→分词→计数的CPU流式块
USE_ALL_GPUS = True
CHUNKS_PER_GPU = 1     # BM25权重计算按nnz切分


In [16]:
# =========================
# Step 3 · Execute: 在 GPU 构建 D–T（三个加权）→ Parquet 落盘
# =========================

import pandas as pd
import numpy as np
import torch
from scipy import sparse

# ---- 1) 读取 Step 2 输出，并解析标签 → 构建 Tag 词表 ----
doc_df = pd.read_parquet(TMP_DIR / "doc_clean.parquet", engine="fastparquet")
N = len(doc_df)

# 解析 tag_str 为列表并展开
tags_series = doc_df["tag_str"].fillna("")
has_tag = tags_series.str.len() > 0

exploded = (
    tags_series[has_tag]
    .str.split(",")
    .explode()
    .str.strip()
    .str.lower()
)
exploded = exploded[exploded != ""]  # 去空

# 统计 df（每个标签在多少文档出现）
# 为确保是“文档级”计数，先拼出 (doc_idx, tag) 对再去重
pairs = pd.DataFrame({
    "doc_idx": doc_df.loc[exploded.index, "doc_idx"].to_numpy(dtype=np.int64),
    "tag": exploded.to_numpy()
})
pairs = pairs.drop_duplicates()

df_counts = pairs.groupby("tag")["doc_idx"].nunique().sort_values(ascending=False)
# 过滤 min_df
df_counts = df_counts[df_counts >= MIN_DF_TAG]
if MAX_TAGS is not None:
    df_counts = df_counts.head(MAX_TAGS)

id2tag = df_counts.index.to_numpy()
T = len(id2tag)
tag2id = {t:i for i,t in enumerate(id2tag)}

print(f"[TAG] kept tags = {T} (min_df >= {MIN_DF_TAG})")

# 保存词表
tag_vocab = pd.DataFrame({
    "tid": np.arange(T, dtype=np.int32),
    "tag": id2tag,
    "df":  df_counts.to_numpy(dtype=np.int64)
})
tag_vocab_path = TMP_DIR / "tag_vocab.parquet"
tag_vocab.to_parquet(tag_vocab_path, engine="fastparquet", index=False)
print(f"[TAG] vocab saved: {tag_vocab_path.as_posix()}")

# ---- 2) 构建 D–T Binary COO（CPU） ----
# 仅保留在词表中的 tag
pairs = pairs[pairs["tag"].isin(tag2id)]
pairs["tid"] = pairs["tag"].map(tag2id).astype(np.int32)
rows_np = pairs["doc_idx"].to_numpy(dtype=np.int64)
cols_np = pairs["tid"].to_numpy(dtype=np.int32)
nnz = len(rows_np)
print(f"[TAG] D–T nnz = {nnz:,}, density = {nnz/(N*max(T,1)):.6e}")

# ---- 3) 转到 GPU，并行计算 TF-IDF 与 PPMI 的“值向量” ----
device_count = torch.cuda.device_count() if USE_ALL_GPUS and torch.cuda.is_available() else 0
devices = [f"cuda:{i}" for i in range(device_count)] if device_count>0 else ["cpu"]
print(f"[GPU] devices used for weighting: {devices}")

# a) 公共统计量（GPU上/或CPU上均可）
df_t = tag_vocab["df"].to_numpy(dtype=np.int64)   # col 度（即 df）
row_deg = np.bincount(rows_np, minlength=N).astype(np.int64)  # 每个 doc 的 kept 标签数

# 转 torch
row_idx_all = torch.from_numpy(rows_np)
col_idx_all = torch.from_numpy(cols_np)
row_deg_t   = torch.from_numpy(row_deg)
col_deg_t   = torch.from_numpy(df_t)
total_pairs = torch.tensor(float(nnz))

# b) 按设备分块
num_chunks = max(1, (device_count or 1) * max(1, CHUNKS_PER_GPU))
chunk_bounds = np.linspace(0, nnz, num_chunks+1, dtype=np.int64)

def _compute_values_chunk(si, ei, dev: str):
    # 输入：全量 row_idx_all/col_idx_all/row_deg_t/col_deg_t
    r = row_idx_all[si:ei].to(dev, non_blocking=True)
    c = col_idx_all[si:ei].to(dev, non_blocking=True)
    row_deg_on_dev = row_deg_t.to(dev)      # 做一次
    col_deg_on_dev = col_deg_t.to(dev)      # 做一次
    rd = row_deg_on_dev[r]                  # 直接在 dev 上索引
    cd = col_deg_on_dev[c]

    # TF-IDF（tf=1）：idf = log((N+1)/(df+1)) + 1  或 log(N/df)
    if TFIDF_IDF_SMOOTH:
        idf = torch.log((torch.tensor(N+1.0, device=dev) / (cd.to(torch.float32)+1.0))) + 1.0
    else:
        idf = torch.log(torch.tensor(float(N), device=dev) / cd.to(torch.float32).clamp_min(1.0))
    tfidf_vals = idf  # tf=1

    # PPMI：pmi = log( total / (r_d * c_t) )
    pmi = torch.log(total_pairs.to(dev) / (rd.to(torch.float32)*cd.to(torch.float32)).clamp_min(PPMI_EPS))
    ppmi_vals = torch.clamp(pmi, min=0.0)

    return tfidf_vals.detach().to("cpu"), ppmi_vals.detach().to("cpu")

# 并行计算
tfidf_list, ppmi_list = [], []
for i in range(num_chunks):
    si, ei = int(chunk_bounds[i]), int(chunk_bounds[i+1])
    dev = devices[i % len(devices)]
    tvi, pvi = _compute_values_chunk(si, ei, dev)
    tfidf_list.append(tvi)
    ppmi_list.append(pvi)

tfidf_vals = torch.cat(tfidf_list).numpy()
ppmi_vals  = torch.cat(ppmi_list).numpy()

# ---- 4) 行归一（L2/L1/None），随后落盘三元组 ----
def _row_normalize_values(rows, vals, num_rows, norm_type: str):
    if norm_type is None:
        return vals
    rows_t = torch.from_numpy(rows).long().to("cuda" if torch.cuda.is_available() else "cpu")
    vals_t = torch.from_numpy(vals).to(rows_t.device, dtype=torch.float32)

    if norm_type.lower() == "l2":
        accum = torch.zeros(num_rows, dtype=torch.float32, device=rows_t.device)
        # 累加平方和
        accum.index_add_(0, rows_t, vals_t*vals_t)
        denom = torch.sqrt(torch.clamp(accum, min=1e-12))
        scaled = vals_t / denom[rows_t]
        return scaled.detach().cpu().numpy()
    elif norm_type.lower() == "l1":
        accum = torch.zeros(num_rows, dtype=torch.float32, device=rows_t.device)
        accum.index_add_(0, rows_t, vals_t.abs())
        denom = torch.clamp(accum, min=1e-12)
        scaled = vals_t / denom[rows_t]
        return scaled.detach().cpu().numpy()
    else:
        return vals

tfidf_vals_norm = _row_normalize_values(rows_np, tfidf_vals, N, ROW_NORM_TFIDF)
ppmi_vals_norm  = _row_normalize_values(rows_np, ppmi_vals,  N, ROW_NORM_PPMI)

# Binary 的值恒为 1；可按需要做行归一（通常不需要），这里保持原样
bin_vals = np.ones_like(rows_np, dtype=np.float32)

# ---- 5) 保存三份矩阵：DT_bin / DT_tfidf / DT_ppmi ----
def _save_triplets(rows, cols, vals, shape, path: Path):
    coo = sparse.coo_matrix((vals, (rows, cols)), shape=shape, dtype=np.float32)
    # 复用 Step 1 的工具（如未定义，可直接 to_parquet）
    from pandas import DataFrame
    pdf = DataFrame({
        "row": coo.row.astype(np.int32),
        "col": coo.col.astype(np.int32),
        "val": coo.data.astype(np.float32),
    })
    pdf.to_parquet(path, engine="fastparquet", index=False)

DT_bin_path   = TMP_DIR / "DT_bin.parquet"
DT_tfidf_path = TMP_DIR / "DT_tfidf.parquet"
DT_ppmi_path  = TMP_DIR / "DT_ppmi.parquet"

shape = (N, T)
_save_triplets(rows_np, cols_np, bin_vals,         shape, DT_bin_path)
_save_triplets(rows_np, cols_np, tfidf_vals_norm,  shape, DT_tfidf_path)
_save_triplets(rows_np, cols_np, ppmi_vals_norm,   shape, DT_ppmi_path)

print("[Step 3] Saved:",
      DT_bin_path.as_posix(), DT_tfidf_path.as_posix(), DT_ppmi_path.as_posix())


[TAG] kept tags = 394 (min_df >= 10)
[TAG] vocab saved: tmp/tag_vocab.parquet
[TAG] D–T nnz = 445,426, density = 2.166853e-03
[GPU] devices used for weighting: ['cuda:0', 'cuda:1']
[Step 3] Saved: tmp/DT_bin.parquet tmp/DT_tfidf.parquet tmp/DT_ppmi.parquet


In [17]:
# =========================
# Step 4 · Params: Text 视图（GPU）—— D–W BM25
# =========================

from pathlib import Path

# 统一TMP路径（沿用你的写法）
TMP_DIR = Path("./tmp")
TMP_DIR.mkdir(exist_ok=True)

# —— 词表筛选 —— 
MIN_DF_W        = 200       # 词在多少文档中出现才保留（与之前保持一致）
MAX_DF_RATIO_W  = 0.50      # 丢弃出现在>50%文档中的“过常见词”
KEEP_TOP_W      = None      # 也可额外设定保留Top-N高df词，None表示不额外截断

# —— 分词 ——（轻量可控；与原先一致的英文token思路）
TOKEN_PATTERN   = r"[a-z0-9_]+"
MIN_TOKEN_LEN   = 2
TO_LOWER        = True

# 可选停用词（保持简洁；你有更全词表也可替换）
STOPWORDS = {
    "the","and","to","of","in","for","on","with","a","is","this","that",
    "it","as","from","by","an","or","be","are","at","we","you","your",
}

# —— BM25 参数 ——（与 IR 常用默认一致）
BM25_K1 = 1.5
BM25_B  = 0.75

# 行归一（便于后续图扩散）
ROW_NORM_BM25 = "l2"   # {"l2","l1",None}

# —— 性能参数 —— 
CHUNK_DOCS   = 50_000  # CSV→文本→分词→计数的CPU流式块
USE_ALL_GPUS = True
CHUNKS_PER_GPU = 1     # BM25权重计算按nnz切分


In [18]:
# =========================
# Step 4 · Execute: 读取text_all→建词表→构建 D–W 计数→GPU上BM25加权→行归一→落盘
# =========================

import re
import numpy as np
import pandas as pd
import torch
from scipy import sparse

# ---------- 0) 读入清洗后的文档 ----------
doc_df = pd.read_parquet(TMP_DIR / "doc_clean.parquet", engine="fastparquet")
N = len(doc_df)
print(f"[TEXT] docs={N:,}")

# ---------- 1) 首遍：扫描文本→构建词表(df) ----------
token_re = re.compile(TOKEN_PATTERN)

def tokenize(s: str):
    if not isinstance(s, str):
        return []
    if TO_LOWER:
        s = s.lower()
    toks = token_re.findall(s)
    toks = [t for t in toks if len(t) >= MIN_TOKEN_LEN and t not in STOPWORDS]
    return toks

df_counter = {}            # word -> df
doc_len = np.zeros(N, dtype=np.int32)

for base in range(0, N, CHUNK_DOCS):
    part = doc_df.iloc[base: base+CHUNK_DOCS]
    unique_in_doc = []   # 收集每个doc的唯一token集合 以计 df
    for i, s in zip(part["doc_idx"].to_numpy(), part["text_all"].tolist()):
        toks = tokenize(s)
        doc_len[i] = len(toks)
        if len(toks) == 0:
            unique_in_doc.append(())
            continue
        uniq = set(toks)
        unique_in_doc.append(uniq)
    # 累加 df
    for uniq in unique_in_doc:
        for w in uniq:
            df_counter[w] = df_counter.get(w, 0) + 1

# 构建词表并过滤
vocab_df = pd.DataFrame({"word": list(df_counter.keys()),
                         "df":   list(df_counter.values())})
vocab_df.sort_values("df", ascending=False, inplace=True)
# 过滤 df / max_df_ratio
vocab_df = vocab_df[vocab_df["df"] >= MIN_DF_W]
vocab_df = vocab_df[vocab_df["df"] <= MAX_DF_RATIO_W * N]
if KEEP_TOP_W is not None:
    vocab_df = vocab_df.head(KEEP_TOP_W)
vocab_df.reset_index(drop=True, inplace=True)

id2word = vocab_df["word"].to_numpy()
df_w    = vocab_df["df"].to_numpy(dtype=np.int64)
W = len(id2word)
word2id = {w:i for i,w in enumerate(id2word)}
print(f"[TEXT] raw_vocab={len(df_counter):,} -> kept(min_df>={MIN_DF_W}, max_df<={MAX_DF_RATIO_W*100:.0f}%)={W:,}")

# 落盘 text_vocab
text_vocab = pd.DataFrame({
    "wid": np.arange(W, dtype=np.int32),
    "word": id2word,
    "df": df_w
})
text_vocab_path = TMP_DIR / "text_vocab.parquet"
text_vocab.to_parquet(text_vocab_path, engine="fastparquet", index=False)
print(f"[TEXT] vocab saved: {text_vocab_path.as_posix()}")

# ---------- 2) 二遍：构建 D–W 计数（三元组） ----------
rows, cols, vals = [], [], []    # tf (term frequency)
row_ptr = []                     # 可选：统计nnz/行

for base in range(0, N, CHUNK_DOCS):
    part = doc_df.iloc[base: base+CHUNK_DOCS]
    for i, s in zip(part["doc_idx"].to_numpy(), part["text_all"].tolist()):
        toks = tokenize(s)
        # 仅保留词表中的词
        tids = [word2id[t] for t in toks if t in word2id]
        if not tids:
            continue
        # 统计该doc的tf
        # 为了速度，使用小字典计数
        cnt = {}
        for t in tids:
            cnt[t] = cnt.get(t, 0) + 1
        # 追加三元组
        for t, f in cnt.items():
            rows.append(i)
            cols.append(t)
            vals.append(f)

rows_np = np.asarray(rows, dtype=np.int64)
cols_np = np.asarray(cols, dtype=np.int32)
tfs_np  = np.asarray(vals, dtype=np.float32)
nnz = len(rows_np)
print(f"[TEXT] D–W nnz={nnz:,}, density={nnz/(N*max(W,1)):.6e}")

# ---------- 3) 在GPU上计算 BM25 权重 ----------
device_count = torch.cuda.device_count() if USE_ALL_GPUS and torch.cuda.is_available() else 0
devices = [f"cuda:{i}" for i in range(device_count)] if device_count>0 else ["cpu"]
print(f"[GPU] devices used for BM25: {devices}")

avgdl = float(doc_len.mean()) if doc_len.mean() > 0 else 1.0

row_idx_all = torch.from_numpy(rows_np)
col_idx_all = torch.from_numpy(cols_np)
tf_all      = torch.from_numpy(tfs_np)

row_len_t = torch.from_numpy(doc_len)     # dl per doc
df_t      = torch.from_numpy(df_w)        # df per word
N_t       = torch.tensor(float(N))

num_chunks = max(1, (device_count or 1) * max(1, CHUNKS_PER_GPU))
bounds = np.linspace(0, nnz, num_chunks+1, dtype=np.int64)

def _bm25_chunk(si, ei, dev: str):
    r = row_idx_all[si:ei].to(dev, non_blocking=True)
    c = col_idx_all[si:ei].to(dev, non_blocking=True)
    tf = tf_all[si:ei].to(dev, non_blocking=True).to(torch.float32)

    # gather dl/df（在CPU上索引再搬/或直接映射到同设备）
    dl = row_len_t[r.cpu()].to(dev, non_blocking=True).to(torch.float32).clamp_min(1.0)
    df = df_t[c.cpu()].to(dev, non_blocking=True).to(torch.float32).clamp_min(1.0)

    # idf = log( (N - df + 0.5) / (df + 0.5) + 1 )
    idf = torch.log(((N_t.to(dev) - df + 0.5) / (df + 0.5)).clamp_min(1e-12) + 1.0)

    # denom = tf + k1*(1 - b + b*dl/avgdl)
    denom = tf + BM25_K1 * (1.0 - BM25_B + BM25_B * (dl / avgdl))
    bm25 = idf * (tf * (BM25_K1 + 1.0) / denom.clamp_min(1e-12))

    return bm25.detach().to("cpu")

bm25_vals_list = []
for i in range(num_chunks):
    si, ei = int(bounds[i]), int(bounds[i+1])
    dev = devices[i % len(devices)]
    bm25_vals_list.append(_bm25_chunk(si, ei, dev))

bm25_vals = torch.cat(bm25_vals_list).numpy()

# ---------- 4) 行归一（L2 / L1 / None） ----------
def _row_normalize_values(rows, vals, num_rows, norm_type: str):
    if norm_type is None:
        return vals
    rows_t = torch.from_numpy(rows).long().to("cuda" if torch.cuda.is_available() else "cpu")
    vals_t = torch.from_numpy(vals).to(rows_t.device, dtype=torch.float32)
    if norm_type.lower() == "l2":
        accum = torch.zeros(num_rows, dtype=torch.float32, device=rows_t.device)
        accum.index_add_(0, rows_t, vals_t*vals_t)
        denom = torch.sqrt(torch.clamp(accum, min=1e-12))
        scaled = vals_t / denom[rows_t]
        return scaled.detach().cpu().numpy()
    elif norm_type.lower() == "l1":
        accum = torch.zeros(num_rows, dtype=torch.float32, device=rows_t.device)
        accum.index_add_(0, rows_t, vals_t.abs())
        denom = torch.clamp(accum, min=1e-12)
        scaled = vals_t / denom[rows_t]
        return scaled.detach().cpu().numpy()
    else:
        return vals

bm25_vals_norm = _row_normalize_values(rows_np, bm25_vals, N, ROW_NORM_BM25)

# ---------- 5) 落盘 DW_bm25（三元组） ----------
DW_bm25_path = TMP_DIR / "DW_bm25.parquet"
coo = sparse.coo_matrix((bm25_vals_norm, (rows_np, cols_np)), shape=(N, W), dtype=np.float32)
pd.DataFrame({
    "row": coo.row.astype(np.int32),
    "col": coo.col.astype(np.int32),
    "val": coo.data.astype(np.float32),
}).to_parquet(DW_bm25_path, engine="fastparquet", index=False)

print("[Step 4] Saved:", DW_bm25_path.as_posix())


[TEXT] docs=521,735
[TEXT] raw_vocab=599,004 -> kept(min_df>=200, max_df<=50%)=5,754
[TEXT] vocab saved: tmp/text_vocab.parquet
[TEXT] D–W nnz=8,140,467, density=2.711624e-03
[GPU] devices used for BM25: ['cuda:0', 'cuda:1']
[Step 4] Saved: tmp/DW_bm25.parquet


In [19]:
# =========================
# Step 5 · Params: GPU 随机游走（类型约束 D–X–D → D-only 语料）
# =========================

from pathlib import Path

# 统一路径
TMP_DIR = Path("./tmp")
TMP_DIR.mkdir(exist_ok=True)

# —— 随机游走参数（高性能取向，与前面一致） ——
RW_WALKS_PER_DOC    = 10       # 每个可用起点D生成多少条游走
RW_L_DOCS_PER_SENT  = 40       # D-only 序列长度（D个数）
RW_SEED_BASE        = 2025     # 基础随机种子（每轮 __iter__ 会 + 偏移）
RW_AVOID_BACKTRACK  = True     # 禁止 D 立即回到上一个 D
RW_RESTART_PROB     = 0.15     # 以概率回到起点 d0（PPR 风格）
RW_X_DEGREE_POW     = -0.5     # X 侧度数惩罚：乘以 (deg(X))^pow 抑制 hub（如 -0.5）
RW_X_NO_REPEAT_LAST = 1        # 最近访问的 X 在下一步禁用（1=不重复上一个 X）

# —— 多 GPU 使用策略 ——
USE_ALL_GPUS   = True          # 使用所有可用 GPU
SPLIT_SHARDS   = 32            # 将起点划分为多少 shard，轮转到各 GPU 上

# —— 调试/抽样输出（生产环境请置0） ——
RW_INSPECT_SAMPLES = 0         # >0 会打印若干条 D-only 样例（不建议太大）


In [20]:
# =========================
# Step 5 · Execute: 读入 D–T / D–W → 构建 GPU 随机游走生成器（在线）
# =========================

import numpy as np
import pandas as pd
import torch
from scipy import sparse

# ---------- 1) 读入准备 ----------
doc_df    = pd.read_parquet(TMP_DIR / "doc_clean.parquet", engine="fastparquet")
tag_vocab = pd.read_parquet(TMP_DIR / "tag_vocab.parquet", engine="fastparquet")
text_vocab= pd.read_parquet(TMP_DIR / "text_vocab.parquet", engine="fastparquet")

N = len(doc_df)
T = len(tag_vocab)
W = len(text_vocab)

def load_csr_from_triplets(path, shape, dtype=np.float32):
    pdf = pd.read_parquet(path, engine="fastparquet")
    coo = sparse.coo_matrix((pdf["val"].astype(dtype), (pdf["row"], pdf["col"])),
                            shape=shape, dtype=dtype)
    return coo.tocsr()

DT_ppmi = load_csr_from_triplets(TMP_DIR / "DT_ppmi.parquet", shape=(N, T))
DW_bm25 = load_csr_from_triplets(TMP_DIR / "DW_bm25.parquet", shape=(N, W))

# ---------- 2) 将 CSR 转为 torch 张量（并在每块 GPU 上各保留一份轻量数据） ----------
def csr_to_torch_rowview(mat: sparse.csr_matrix, device: torch.device):
    """把 CSR 的 (indptr, indices, data) 移到指定 device；用于快速行切片。"""
    mat = mat.tocsr()
    indptr = torch.from_numpy(mat.indptr.astype(np.int64)).to(device)
    indices= torch.from_numpy(mat.indices.astype(np.int64)).to(device)
    data   = torch.from_numpy(mat.data.astype(np.float32)).to(device)
    return indptr, indices, data

def csr_transpose(mat: sparse.csr_matrix) -> sparse.csr_matrix:
    return mat.transpose().tocsr()

devices = []
if torch.cuda.is_available() and USE_ALL_GPUS:
    devices = [torch.device(f"cuda:{i}") for i in range(torch.cuda.device_count())]
if not devices:
    devices = [torch.device("cpu")]
print(f"[RW] devices: {[str(d) for d in devices]}")

# 为每个 device 构造两视图的 D->X / X->D 访问结构
DX_tag_dev, XD_tag_dev = [], []
DX_txt_dev, XD_txt_dev = [], []

for dev in devices:
    DX_tag_dev.append(csr_to_torch_rowview(DT_ppmi, dev))
    XD_tag_dev.append(csr_to_torch_rowview(csr_transpose(DT_ppmi), dev))
    DX_txt_dev.append(csr_to_torch_rowview(DW_bm25, dev))
    XD_txt_dev.append(csr_to_torch_rowview(csr_transpose(DW_bm25), dev))

# 预先计算 row/col 度数（在 CPU 上，便于快速判断起点）
degD_tag = np.diff(DT_ppmi.indptr)
degD_txt = np.diff(DW_bm25.indptr)

start_tag = np.where(degD_tag > 0)[0].astype(np.int64)
start_txt = np.where(degD_txt > 0)[0].astype(np.int64)

print(f"[RW-tag] starts={len(start_tag):,}, target_walks≈{len(start_tag)*RW_WALKS_PER_DOC:,}, L={RW_L_DOCS_PER_SENT}")
print(f"[RW-text] starts={len(start_txt):,}, target_walks≈{len(start_txt)*RW_WALKS_PER_DOC:,}, L={RW_L_DOCS_PER_SENT}")

# ---------- 3) 工具函数：行邻接访问 + 加权采样 ----------
def row_neighbors(indptr: torch.Tensor, indices: torch.Tensor, data: torch.Tensor, r: torch.Tensor):
    """返回行 r 的 (cols, weights) 视图（不复制）。r 是标量 LongTensor。"""
    a = indptr[r]
    b = indptr[r+1]
    if (b - a).item() <= 0:
        return None, None
    sl = slice(a.item(), b.item())
    return indices[sl], data[sl]

def sample_pos_by_weights(w: torch.Tensor, g: torch.Generator) -> int:
    """给定非负权重 w，按概率采样返回位置（-1 表示失败）。"""
    # 安全处理
    if w is None or w.numel() == 0:
        return -1
    w = torch.clamp(w, min=0)
    s = w.sum()
    if not torch.isfinite(s) or s.item() <= 0:
        return -1
    # 前缀和 + 二分
    cdf = torch.cumsum(w, dim=0)
    u = torch.rand((), generator=g, device=w.device) * cdf[-1]
    pos = torch.searchsorted(cdf, u, right=False).item()
    if pos >= cdf.numel(): pos = cdf.numel()-1
    return pos

# ---------- 4) 在线语料生成器（两视图各一个），支持多 GPU 分片 ----------
class TorchWalkCorpus:
    """
    每次 __iter__：
      1) 重新洗牌起点并分 shard；
      2) 轮转到各 GPU，使用对应的 DX/XD 结构做 D–X–D 随机游走；
      3) 产出 D-only 序列（list[str]）。
    """
    def __init__(self, starts_np: np.ndarray, DX_views, XD_views, view_name: str, base_seed: int):
        self.starts_np = starts_np
        self.DX_views = DX_views     # list of (indptr, indices, data) per device
        self.XD_views = XD_views
        self.view_name = view_name
        self.base_seed = base_seed
        self._iters = 0

    def __len__(self):
        return int(len(self.starts_np) * RW_WALKS_PER_DOC)

    def __iter__(self):
        self._iters += 1
        rng = np.random.default_rng(self.base_seed + 31*self._iters)
        starts = self.starts_np.copy()
        rng.shuffle(starts)

        # 分 shard，轮转设备
        shards = np.array_split(starts, max(1, SPLIT_SHARDS))
        devs = devices

        for shard_id, shard in enumerate(shards):
            dev = devs[shard_id % len(devs)]
            DX = self.DX_views[shard_id % len(devs)]
            XD = self.XD_views[shard_id % len(devs)]
            indptr_D, indices_D, data_D = DX
            indptr_X, indices_X, data_X = XD

            # 每个 shard 使用独立的 torch RNG
            g = torch.Generator(device=dev)
            g.manual_seed(self.base_seed + 7919 * (self._iters + shard_id))

            # 预备 X 侧度数惩罚
            if abs(RW_X_DEGREE_POW) > 1e-12:
                # X 的度 = XD 行度
                x_deg = (indptr_X[1:] - indptr_X[:-1]).to(torch.float32)
                x_factor = torch.clamp(x_deg, min=1.0).pow(RW_X_DEGREE_POW)
            else:
                x_factor = None

            # 遍历 shard 中的起点
            for si, d0 in enumerate(shard):
                d0_t = torch.tensor(int(d0), dtype=torch.long, device=dev)
                for _ in range(RW_WALKS_PER_DOC):
                    seq = [int(d0)]
                    prev_d = None
                    cur_d  = int(d0)
                    last_x = -1  # 最近一次访问的 X（-1 表示无）

                    for _step in range(RW_L_DOCS_PER_SENT-1):
                        # D -> X
                        r = torch.tensor(cur_d, dtype=torch.long, device=dev)
                        x_cols, x_w = row_neighbors(indptr_D, indices_D, data_D, r)
                        if x_cols is None:
                            break

                        # 权重：边权 × X度数惩罚
                        w = x_w.clone()
                        if x_factor is not None:
                            w = w * x_factor[x_cols]

                        # 禁止重复上一个 X
                        if RW_X_NO_REPEAT_LAST > 0 and last_x >= 0:
                            mask = (x_cols == last_x)
                            if mask.any():
                                w[mask] = 0.0

                        pos_x = sample_pos_by_weights(w, g)
                        if pos_x < 0:
                            break
                        x = int(x_cols[pos_x].item())

                        # X -> D
                        # 通过 XD（X 行）找到相邻 D
                        xr = torch.tensor(x, dtype=torch.long, device=dev)
                        d_rows, d_w = row_neighbors(indptr_X, indices_X, data_X, xr)
                        if d_rows is None:
                            break

                        # 禁回头
                        if RW_AVOID_BACKTRACK and prev_d is not None and d_rows.numel() > 1:
                            mask = (d_rows == prev_d)
                            if mask.any():
                                d_w = d_w.clone()
                                d_w[mask] = 0.0

                        pos_d = sample_pos_by_weights(d_w, g)
                        if pos_d < 0:
                            break
                        next_d = int(d_rows[pos_d].item())

                        # 重启
                        if torch.rand((), generator=g, device=dev).item() < RW_RESTART_PROB:
                            next_d = int(d0)

                        seq.append(next_d)
                        prev_d, cur_d = cur_d, next_d
                        last_x = x

                    if len(seq) >= 2:
                        # 产出 D-only 序列（str token）
                        yield [str(s) for s in seq]

# ---------- 5) 实例化两视图的生成器 ----------
tag_corpus_torch  = TorchWalkCorpus(start_tag, DX_tag_dev, XD_tag_dev, view_name="tag",  base_seed=RW_SEED_BASE+11)
text_corpus_torch = TorchWalkCorpus(start_txt, DX_txt_dev, XD_txt_dev, view_name="text", base_seed=RW_SEED_BASE+23)

print(f"[RW] tag ~{len(tag_corpus_torch):,} sentences; text ~{len(text_corpus_torch):,} sentences; L={RW_L_DOCS_PER_SENT}")

# ---------- 6) 可选：抽样打印若干条（便于快速肉眼质检；跑完可以删除） ----------
if RW_INSPECT_SAMPLES > 0:
    import itertools
    print("\n[INSPECT] sample tag sequences:")
    for s in itertools.islice(iter(tag_corpus_torch), RW_INSPECT_SAMPLES):
        print("  ", s[:min(len(s), 20)])
    print("\n[INSPECT] sample text sequences:")
    for s in itertools.islice(iter(text_corpus_torch), RW_INSPECT_SAMPLES):
        print("  ", s[:min(len(s), 20)])

# ---------- 7) 落盘：保存随机游走参数（便于 Step 6 自给自足重建生成器） ----------
pd.DataFrame([{
    "RW_WALKS_PER_DOC": RW_WALKS_PER_DOC,
    "RW_L_DOCS_PER_SENT": RW_L_DOCS_PER_SENT,
    "RW_SEED_BASE": RW_SEED_BASE,
    "RW_AVOID_BACKTRACK": RW_AVOID_BACKTRACK,
    "RW_RESTART_PROB": RW_RESTART_PROB,
    "RW_X_DEGREE_POW": RW_X_DEGREE_POW,
    "RW_X_NO_REPEAT_LAST": RW_X_NO_REPEAT_LAST,
    "USE_ALL_GPUS": USE_ALL_GPUS,
    "SPLIT_SHARDS": SPLIT_SHARDS,
}]).to_parquet(TMP_DIR / "rw_params.parquet", engine="fastparquet", index=False)

print("[Step 5] Online corpus builders ready. Params saved:", (TMP_DIR / "rw_params.parquet").as_posix())


[RW] devices: ['cuda:0', 'cuda:1']
[RW-tag] starts=214,585, target_walks≈2,145,850, L=40
[RW-text] starts=416,697, target_walks≈4,166,970, L=40
[RW] tag ~2,145,850 sentences; text ~4,166,970 sentences; L=40
[Step 5] Online corpus builders ready. Params saved: tmp/rw_params.parquet


In [25]:
# 更清晰的链式打印（可运行后删除）
import numpy as np
import pandas as pd
from scipy import sparse
from pathlib import Path

TMP_DIR = Path("./tmp")
doc_df    = pd.read_parquet(TMP_DIR / "doc_clean.parquet", engine="fastparquet")
tag_vocab = pd.read_parquet(TMP_DIR / "tag_vocab.parquet", engine="fastparquet")
text_vocab= pd.read_parquet(TMP_DIR / "text_vocab.parquet", engine="fastparquet")

def load_csr(path, shape, dtype=np.float32):
    df = pd.read_parquet(path, engine="fastparquet")
    coo = sparse.coo_matrix((df["val"].astype(dtype), (df["row"], df["col"])), shape=shape, dtype=dtype)
    return coo.tocsr()

DT_ppmi = load_csr(TMP_DIR/"DT_ppmi.parquet", shape=(len(doc_df), len(tag_vocab)))
DW_bm25 = load_csr(TMP_DIR/"DW_bm25.parquet", shape=(len(doc_df), len(text_vocab)))

rw = pd.read_parquet(TMP_DIR / "rw_params.parquet", engine="fastparquet").iloc[0]
RW_RESTART_PROB     = float(rw["RW_RESTART_PROB"])
RW_AVOID_BACKTRACK  = bool(rw["RW_AVOID_BACKTRACK"])
RW_X_DEGREE_POW     = float(rw["RW_X_DEGREE_POW"])
RW_X_NO_REPEAT_LAST = int(rw["RW_X_NO_REPEAT_LAST"])

def _row_neighbors_csr(mat, r):
    a,b = mat.indptr[r], mat.indptr[r+1]
    if a==b: return None, None
    return mat.indices[a:b], mat.data[a:b]

def _sample_by_weights(w, rng):
    w = np.asarray(w, dtype=np.float64)
    tot = w.sum()
    if not np.isfinite(tot) or tot<=0: return None
    cdf = np.cumsum(w)
    pos = int(np.searchsorted(cdf, rng.random()*tot, side="left"))
    if pos >= cdf.size: pos = cdf.size-1
    return pos

def print_chain(name, DX, XD, id2x, x_lab, n_samples=3, L_docs=8, seed=123):
    rng = np.random.default_rng(seed)
    starts = np.where(np.diff(DX.indptr) > 0)[0]
    x_deg = np.diff(XD.indptr).astype(np.float64)
    x_factor = np.power(np.maximum(x_deg,1.0), RW_X_DEGREE_POW) if abs(RW_X_DEGREE_POW)>1e-12 else None

    print(f"\n[CHECK-{name}] {n_samples} 条（目标 D 长度≈{L_docs}）")
    for k in range(n_samples):
        d0 = int(rng.choice(starts))
        seqD = [d0]; seqX = []
        prev_d, cur_d, last_x = None, d0, None
        for _ in range(L_docs-1):
            x_cols, x_w = _row_neighbors_csr(DX, cur_d)
            if x_cols is None: break
            w = x_w.copy()
            if x_factor is not None: w *= x_factor[x_cols]
            if RW_X_NO_REPEAT_LAST>0 and last_x is not None:
                w = w.copy(); w[x_cols==last_x] = 0.0
            p = _sample_by_weights(w, rng); 
            if p is None: break
            x = int(x_cols[p]); seqX.append(x)

            d_rows, d_w = _row_neighbors_csr(XD, x)
            if d_rows is None: break
            if RW_AVOID_BACKTRACK and prev_d is not None and d_rows.size>1:
                d_w = d_w.copy(); d_w[d_rows==prev_d] = 0.0
            p2 = _sample_by_weights(d_w, rng)
            if p2 is None: break
            next_d = int(d_rows[p2])

            if rng.random() < RW_RESTART_PROB:
                next_d = d0
            seqD.append(next_d)
            prev_d, cur_d, last_x = cur_d, next_d, x

        # 连续打印：D0 --X[x0]--> D1 --X[x1]--> D2 ...
        chain = f"D({seqD[0]}/Id={int(doc_df.loc[seqD[0],'Id'])})"
        for i, x in enumerate(seqX):
            xname = id2x[x]
            d_id  = int(doc_df.loc[seqD[i+1],"Id"])
            chain += f" --{x_lab}[{xname}]--> D({seqD[i+1]}/Id={d_id})"
        print(f"  示例 #{k+1}\n  {chain}")

print_chain("tag",  DT_ppmi, DT_ppmi.transpose().tocsr(), tag_vocab["tag"].to_numpy(),  "T", n_samples=3, L_docs=8)
print_chain("text", DW_bm25, DW_bm25.transpose().tocsr(), text_vocab["word"].to_numpy(), "W", n_samples=3, L_docs=8)
print("\n[OK] 更清晰的链式打印完成。")



[CHECK-tag] 3 条（目标 D 长度≈8）
  示例 #1
  D(4911/Id=14578) --T[arts and entertainment]--> D(77827/Id=1269722)
  示例 #2
  D(335565/Id=5161079) --T[beginner]--> D(376473/Id=5751643) --T[news]--> D(399727/Id=6139562)
  示例 #3
  D(8369/Id=37754) --T[standardized testing]--> D(460708/Id=7140595)

[CHECK-text] 3 条（目标 D 长度≈8）
  示例 #1
  D(7249/Id=31415) --W[test]--> D(135089/Id=2007600) --W[code]--> D(417767/Id=6437396) --W[official]--> D(504488/Id=7819842) --W[face]--> D(117467/Id=1767210) --W[dataset]--> D(423370/Id=6539906)
  示例 #2
  D(354554/Id=5398358) --W[infectious]--> D(48176/Id=835414) --W[chest]--> D(81804/Id=1317048) --W[45]--> D(345889/Id=5259310) --W[12]--> D(230416/Id=3500693) --W[lyft]--> D(58777/Id=973364) --W[valid]--> D(104202/Id=1585399) --W[tfrecords]--> D(80985/Id=1307594)
  示例 #3
  D(236152/Id=3585148) --W[million]--> D(236152/Id=3585148) --W[ranked]--> D(111755/Id=1688726) --W[2020]--> D(236152/Id=3585148) --W[various]--> D(341366/Id=5251121) --W[trends]--> D(236152/Id=3585148

In [2]:
# =========================
# Step 6 · Params: PyTorch 多GPU WS-SGNS（Skip-gram + Negative Sampling）
# =========================

from pathlib import Path

# 统一路径
TMP_DIR = Path("./tmp")
TMP_DIR.mkdir(exist_ok=True)

# —— SGNS 超参（高性能但注意显存）——
SGNS_DIM        = 256        # 嵌入维度（可提到384，但显存↑）
SGNS_WINDOW     = 5          # 最大上下文窗口
SGNS_NEGATIVE   = 12        # 每对正样本的负样本数
SGNS_EPOCHS     = 4          # 轮次（可上调到6-8获取更好向量）
SGNS_LR         = 0.05       # SGD 学习率（稳定且显存友好）
SGNS_CLIP_NORM  = 2.0        # 梯度裁剪（稳定训练）

# 批量（以“正样本对数”为计量单位）
PAIRS_BATCH_SIZE = 10240     # 每次反向的正样本对数（显存关键；不够就下调到4096）

# 语料规模控制（为防止epoch过长，保留可选上限；None=全量）
MAX_SENTS_PER_EPOCH_TAG  = None     # 例：100_000
MAX_SENTS_PER_EPOCH_TEXT = None

# 负采样分布：基于 D 的度数（来自二部图行度）^0.75
NS_POWER = 0.75

# 多 GPU
USE_ALL_GPUS = True          # DataParallel 覆盖所有 GPU
RNG_BASE_SEED = 2025         # 随机种子基

In [ ]:
# =========================
# Step 6 · Execute: 读入矩阵与参数 → 重建在线语料 → 训练 WS-SGNS → 落盘 Z_tag / Z_text
# =========================

import math, gc, time, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scipy import sparse

# ---------- 0) 读入 Step 2/3/4/5 的中间件 ----------
doc_df    = pd.read_parquet(TMP_DIR / "doc_clean.parquet", engine="fastparquet")
tag_vocab = pd.read_parquet(TMP_DIR / "tag_vocab.parquet", engine="fastparquet")
text_vocab= pd.read_parquet(TMP_DIR / "text_vocab.parquet", engine="fastparquet")
rw_params = pd.read_parquet(TMP_DIR / "rw_params.parquet", engine="fastparquet").iloc[0]

N = len(doc_df);  T = len(tag_vocab);  W = len(text_vocab)

def load_csr(path, shape, dtype=np.float32):
    df = pd.read_parquet(path, engine="fastparquet")
    coo = sparse.coo_matrix((df["val"].astype(dtype), (df["row"], df["col"])), shape=shape, dtype=dtype)
    return coo.tocsr()

DT_ppmi = load_csr(TMP_DIR / "DT_ppmi.parquet", shape=(N, T))
DW_bm25 = load_csr(TMP_DIR / "DW_bm25.parquet", shape=(N, W))

# ---------- 1) 复用 Step 5 的在线随机游走（必要代码最小复制，确保本步可独立运行） ----------
def csr_to_rowview_torch(mat: sparse.csr_matrix, device: torch.device):
    mat = mat.tocsr()
    indptr = torch.from_numpy(mat.indptr.astype(np.int64)).to(device)
    indices= torch.from_numpy(mat.indices.astype(np.int64)).to(device)
    data   = torch.from_numpy(mat.data.astype(np.float32)).to(device)
    return indptr, indices, data

def csr_T(mat: sparse.csr_matrix) -> sparse.csr_matrix:
    return mat.transpose().tocsr()

# 设备
devices = []
if torch.cuda.is_available() and USE_ALL_GPUS:
    devices = [torch.device(f"cuda:{i}") for i in range(torch.cuda.device_count())]
if not devices:
    devices = [torch.device("cpu")]
print(f"[SGNS] using devices: {[str(d) for d in devices]}")

# 将 D–X 与 X–D 行视图分别放到各GPU（轻量）
DX_tag_dev = []; XD_tag_dev = []
DX_txt_dev = []; XD_txt_dev = []
for dev in devices:
    DX_tag_dev.append(csr_to_rowview_torch(DT_ppmi, dev))
    XD_tag_dev.append(csr_to_rowview_torch(csr_T(DT_ppmi), dev))
    DX_txt_dev.append(csr_to_rowview_torch(DW_bm25, dev))
    XD_txt_dev.append(csr_to_rowview_torch(csr_T(DW_bm25), dev))

# 起点集合（有邻接的文档）
degD_tag = np.diff(DT_ppmi.indptr)
degD_txt = np.diff(DW_bm25.indptr)
start_tag = np.where(degD_tag > 0)[0].astype(np.int64)
start_txt = np.where(degD_txt > 0)[0].astype(np.int64)

# 读回 RW 参数
RW_WALKS_PER_DOC    = int(rw_params["RW_WALKS_PER_DOC"])
RW_L_DOCS_PER_SENT  = int(rw_params["RW_L_DOCS_PER_SENT"])
RW_SEED_BASE        = int(rw_params["RW_SEED_BASE"])
RW_AVOID_BACKTRACK  = bool(rw_params["RW_AVOID_BACKTRACK"])
RW_RESTART_PROB     = float(rw_params["RW_RESTART_PROB"])
RW_X_DEGREE_POW     = float(rw_params["RW_X_DEGREE_POW"])
RW_X_NO_REPEAT_LAST = int(rw_params["RW_X_NO_REPEAT_LAST"])

# 行邻接 + 采样
def row_neighbors(indptr, indices, data, r: torch.Tensor):
    a = indptr[r]; b = indptr[r+1]
    if (b-a).item() <= 0: return None, None
    sl = slice(a.item(), b.item())
    return indices[sl], data[sl]

def sample_pos_by_weights(w: torch.Tensor, g: torch.Generator) -> int:
    if w is None or w.numel()==0: return -1
    w = torch.clamp(w, min=0)
    s = w.sum()
    if not torch.isfinite(s) or s.item()<=0: return -1
    cdf = torch.cumsum(w, dim=0)
    u = torch.rand((), generator=g, device=w.device) * cdf[-1]
    pos = torch.searchsorted(cdf, u, right=False).item()
    return min(pos, cdf.numel()-1)

# 在线语料（与 Step 5 保持一致）
class TorchWalkCorpus:
    def __init__(self, starts_np, DX_views, XD_views, base_seed, split_shards=32, view_name=""):
        self.starts_np = starts_np
        self.DX_views = DX_views; self.XD_views = XD_views
        self.base_seed = base_seed; self.split_shards = max(1, split_shards)
        self._iters = 0; self.view_name = view_name
    def __len__(self):
        return int(len(self.starts_np) * RW_WALKS_PER_DOC)
    def __iter__(self):
        self._iters += 1
        rng = np.random.default_rng(RW_SEED_BASE + 31*self._iters)
        starts = self.starts_np.copy(); rng.shuffle(starts)
        shards = np.array_split(starts, self.split_shards)
        devs = devices
        for shard_id, shard in enumerate(shards):
            dev = devs[shard_id % len(devs)]
            DX = self.DX_views[shard_id % len(devs)]
            XD = self.XD_views[shard_id % len(devs)]
            indptr_D, indices_D, data_D = DX
            indptr_X, indices_X, data_X = XD
            g = torch.Generator(device=dev)
            g.manual_seed(self.base_seed + 7919*(self._iters + shard_id))
            x_factor = None
            if abs(RW_X_DEGREE_POW) > 1e-12:
                x_deg = (indptr_X[1:]-indptr_X[:-1]).to(torch.float32)
                x_factor = torch.clamp(x_deg, min=1.0).pow(RW_X_DEGREE_POW)
            for d0 in shard:
                for _ in range(RW_WALKS_PER_DOC):
                    seq = [int(d0)]
                    prev_d=None; cur_d=int(d0); last_x=-1
                    for _step in range(RW_L_DOCS_PER_SENT-1):
                        r = torch.tensor(cur_d, dtype=torch.long, device=dev)
                        x_cols, x_w = row_neighbors(indptr_D, indices_D, data_D, r)
                        if x_cols is None: break
                        w = x_w.clone()
                        if x_factor is not None: w = w * x_factor[x_cols]
                        if RW_X_NO_REPEAT_LAST>0 and last_x>=0:
                            mask = (x_cols==last_x)
                            if mask.any(): w[mask]=0.0
                        pos_x = sample_pos_by_weights(w, g)
                        if pos_x < 0: break
                        x = int(x_cols[pos_x].item())

                        xr = torch.tensor(x, dtype=torch.long, device=dev)
                        d_rows, d_w = row_neighbors(indptr_X, indices_X, data_X, xr)
                        if d_rows is None: break
                        if RW_AVOID_BACKTRACK and prev_d is not None and d_rows.numel()>1:
                            m = (d_rows==prev_d)
                            if m.any(): d_w=d_w.clone(); d_w[m]=0.0
                        pos_d = sample_pos_by_weights(d_w, g)
                        if pos_d < 0: break
                        next_d = int(d_rows[pos_d].item())

                        if torch.rand((), generator=g, device=dev).item() < RW_RESTART_PROB:
                            next_d = int(d0)

                        seq.append(next_d)
                        prev_d, cur_d, last_x = cur_d, next_d, x
                    if len(seq)>=2: yield [str(s) for s in seq]

# 两个视图的语料
tag_corpus  = TorchWalkCorpus(start_tag, DX_tag_dev, XD_tag_dev, base_seed=RW_SEED_BASE+11, split_shards=64, view_name="tag")
text_corpus = TorchWalkCorpus(start_txt, DX_txt_dev, XD_txt_dev, base_seed=RW_SEED_BASE+23, split_shards=64, view_name="text")

# ---------- 2) SGNS 模型 ----------
class SGNS(nn.Module):
    def __init__(self, vocab_size: int, dim: int):
        super().__init__()
        self.in_emb  = nn.Embedding(vocab_size, dim, sparse=False)
        self.out_emb = nn.Embedding(vocab_size, dim, sparse=False)
        nn.init.uniform_(self.in_emb.weight,  -0.5/dim, 0.5/dim)
        nn.init.uniform_(self.out_emb.weight, -0.5/dim, 0.5/dim)
    def forward(self, center, pos, neg):
        # center: [B], pos: [B], neg: [B,K]
        v = self.in_emb(center)        # [B,d]
        u = self.out_emb(pos)          # [B,d]
        pos_logit = torch.sum(v*u, dim=1)                  # [B]
        # 负样本
        neg_u = self.out_emb(neg)      # [B,K,d]
        neg_logit = torch.einsum("bd,bkd->bk", v, neg_u)   # [B,K]
        # loss = - (log σ(pos) + sum log σ(-neg))
        pos_loss = torch.nn.functional.softplus(-pos_logit)         # -log σ(x) = softplus(-x)
        neg_loss = torch.nn.functional.softplus(neg_logit).sum(dim=1)  # -log σ(-x) = softplus(x)
        return (pos_loss + neg_loss).mean().unsqueeze(0)

def use_all_gpus(module: nn.Module) -> nn.Module:
    if torch.cuda.is_available() and USE_ALL_GPUS and torch.cuda.device_count()>1:
        return nn.DataParallel(module)  # 简洁起见
    return module

# ---------- 3) 负采样分布（基于行度 ^0.75） ----------
def build_ns_dist_from_deg(deg: np.ndarray, power=0.75):
    p = np.power(np.maximum(deg, 1), power).astype(np.float64)
    p = p / p.sum()
    return p

ns_dist_tag  = build_ns_dist_from_deg(degD_tag, power=NS_POWER)
ns_dist_text = build_ns_dist_from_deg(degD_txt, power=NS_POWER)

# ---------- 4) 语料 → 正样本对迭代器（滑窗） ----------
def iter_pairs_from_corpus(corpus, window: int, max_sents: int = None, seed: int = 0):
    rng = random.Random(seed)
    sent_count = 0
    for sent in corpus:
        # sent: list[str] of doc_idx
        if max_sents is not None and sent_count >= max_sents: break
        s = [int(x) for x in sent]
        L = len(s)
        for i in range(L):
            # 动态窗口：1..window
            w = rng.randint(1, window)
            l = max(0, i - w); r = min(L - 1, i + w)
            for j in range(l, r + 1):
                if j == i: continue
                yield s[i], s[j]   # (center, context)
        sent_count += 1

# ---------- 5) 小批量采样器（输出张量） ----------
def batch_pairs_and_negs(pair_iter, batch_size_pairs: int, negK: int, ns_dist: np.ndarray, device: torch.device):
    centers = []
    contexts= []
    for c,x in pair_iter:
        centers.append(c); contexts.append(x)
        if len(centers) >= batch_size_pairs:
            B = len(centers)
            # 负样本在 CPU 采样
            negs = np.random.choice(len(ns_dist), size=(B, negK), p=ns_dist)

            # 关键修复：先在 CPU 构张量，再 .to(device, non_blocking=True)
            centers_cpu = torch.tensor(centers, dtype=torch.long)           # CPU
            contexts_cpu= torch.tensor(contexts, dtype=torch.long)          # CPU
            negs_cpu    = torch.tensor(negs,    dtype=torch.long)           # CPU

            # 可选：固定有 CUDA 时，用 pinned memory 提升 H2D 速度
            if torch.cuda.is_available():
                centers_cpu = centers_cpu.pin_memory()
                contexts_cpu= contexts_cpu.pin_memory()
                negs_cpu    = negs_cpu.pin_memory()

            centers_t = centers_cpu.to(device, non_blocking=True)
            contexts_t= contexts_cpu.to(device, non_blocking=True)
            negs_t    = negs_cpu.to(device, non_blocking=True)

            yield centers_t, contexts_t, negs_t
            centers.clear(); contexts.clear()

    # 尾部
    if centers:
        B = len(centers)
        negs = np.random.choice(len(ns_dist), size=(B, negK), p=ns_dist)

        centers_cpu = torch.tensor(centers, dtype=torch.long)
        contexts_cpu= torch.tensor(contexts, dtype=torch.long)
        negs_cpu    = torch.tensor(negs,    dtype=torch.long)

        if torch.cuda.is_available():
            centers_cpu = centers_cpu.pin_memory()
            contexts_cpu= contexts_cpu.pin_memory()
            negs_cpu    = negs_cpu.pin_memory()

        centers_t = centers_cpu.to(device, non_blocking=True)
        contexts_t= contexts_cpu.to(device, non_blocking=True)
        negs_t    = negs_cpu.to(device, non_blocking=True)

        yield centers_t, contexts_t, negs_t


# ---------- 6) 训练函数（单视图） ----------
def train_view(view_name: str, degD: np.ndarray, ns_dist: np.ndarray, corpus, max_sents: int, out_path: Path):
    torch.manual_seed(RNG_BASE_SEED + (11 if view_name=="tag" else 23))
    base_device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

    model = SGNS(vocab_size=N, dim=SGNS_DIM)
    model = use_all_gpus(model).to(base_device)

    optimizer = torch.optim.SGD(model.parameters(), lr=SGNS_LR)

    print(f"[Train-{view_name}] epochs={SGNS_EPOCHS}, dim={SGNS_DIM}, window={SGNS_WINDOW}, neg={SGNS_NEGATIVE}, batch_pairs={PAIRS_BATCH_SIZE}")
    for ep in range(1, SGNS_EPOCHS+1):
        t0 = time.time()
        # 为每个 epoch 重新构造 pairs 迭代器（在线语料）
        pair_iter = iter_pairs_from_corpus(
            corpus=corpus,
            window=SGNS_WINDOW,
            max_sents=max_sents,
            seed=RNG_BASE_SEED + ep
        )
        total_pairs = 0
        total_loss = 0.0
        model.train()
        for centers_t, contexts_t, negs_t in batch_pairs_and_negs(
                pair_iter, PAIRS_BATCH_SIZE, SGNS_NEGATIVE, ns_dist, device=base_device):
            optimizer.zero_grad(set_to_none=True)
            loss = model(centers_t, contexts_t, negs_t)
            # DataParallel 会把每卡的标量 loss 聚合成 [num_gpus] 向量，这里再压成标量
            if hasattr(loss, "dim") and loss.dim() != 0:
                loss = loss.mean()

            loss.backward()
            # 梯度裁剪
            if SGNS_CLIP_NORM is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), SGNS_CLIP_NORM)
            optimizer.step()
            total_pairs += centers_t.size(0)
            total_loss  += float(loss.detach().item()) * centers_t.size(0)
        dt = time.time() - t0
        if total_pairs == 0:
            print(f"[Train-{view_name}] epoch {ep}: no pairs produced (max_sents too small?)")
        else:
            print(f"[Train-{view_name}] epoch {ep}: pairs={total_pairs:,}, avg_loss={total_loss/total_pairs:.4f}, time={dt:.1f}s")

    # 提取输入嵌入作为最终向量（也可 in+out 平均；这里取 in 并做 L2 归一）
    if isinstance(model, nn.DataParallel):
        E = model.module.in_emb.weight.detach().cpu().numpy()
    else:
        E = model.in_emb.weight.detach().cpu().numpy()
    # 仅对有邻接的文档有效；其余保持零向量
    Z = E.astype(np.float32, copy=True)
    nrm = np.linalg.norm(Z, axis=1, keepdims=True)
    mask = (nrm[:,0] > 0)
    Z[mask] = Z[mask] / nrm[mask]

    # 落盘（与旧管线一致）
    emb_df = pd.DataFrame(Z, columns=[f"f{i}" for i in range(Z.shape[1])])
    emb_df.insert(0, "doc_idx", np.arange(N, dtype=np.int64))
    emb_df.to_parquet(out_path, engine="fastparquet", index=False)
    covered = int(mask.sum())
    print(f"[Train-{view_name}] saved {out_path.name}; covered_docs={covered}/{N} ({covered/max(N,1):.1%})")

    # 显存回收
    del model, optimizer; gc.collect()
    torch.cuda.empty_cache()

# ---------- 7) 先训练 tag，再训练 text（顺序执行，避免峰值显存叠加） ----------
train_view(
    view_name="tag",
    degD=degD_tag,
    ns_dist=ns_dist_tag,
    corpus=tag_corpus,
    max_sents=MAX_SENTS_PER_EPOCH_TAG,
    out_path=TMP_DIR / "Z_tag.parquet"
)

train_view(
    view_name="text",
    degD=degD_txt,
    ns_dist=ns_dist_text,
    corpus=text_corpus,
    max_sents=MAX_SENTS_PER_EPOCH_TEXT,
    out_path=TMP_DIR / "Z_text.parquet"
)

print("[Step 6] Done. Embeddings saved to Z_tag.parquet / Z_text.parquet")


[SGNS] using devices: ['cuda:0', 'cuda:1']
[Train-tag] epochs=4, dim=256, window=5, neg=12, batch_pairs=10240
[Train-tag] epoch 1: pairs=28,994,522, avg_loss=9.0109, time=5859.3s
[Train-tag] epoch 2: pairs=29,003,471, avg_loss=9.0109, time=5396.1s
[Train-tag] epoch 3: pairs=28,918,395, avg_loss=9.0109, time=5327.6s
[Train-tag] epoch 4: pairs=28,941,352, avg_loss=9.0109, time=5329.6s
[Train-tag] saved Z_tag.parquet; covered_docs=521735/521735 (100.0%)
[Train-text] epochs=4, dim=256, window=5, neg=12, batch_pairs=10240
[Train-text] epoch 1: pairs=300,725,185, avg_loss=9.0109, time=41298.2s


KeyboardInterrupt: 

In [7]:
# =========================
# Step 7 · Params: ANN 构图（Tag + Text）
# =========================
from pathlib import Path

TMP_DIR = Path("./tmp").resolve()

# 近邻与检索批量
K_NEIGHBORS         = 50          # 每个文档保留的近邻数（不含自环）
SEARCH_BATCH_Q      = 8192        # 查询批量（越大越快，显存越大）
SAVE_PART_EDGES     = 2_000_000   # 每个分片最多保存的边条数（控制单文件大小）

# FAISS 选项（GPU 优先，CPU 兜底；Flat-IP=精确检索，速度快/显存足够）
FAISS_USE_GPU       = True
FAISS_INDEX_TYPE    = "flat_ip"   # "flat_ip" | "ivf_flat"（如需更大规模可切 IVF）
IVF_NLIST           = 4096        # 仅在 ivf_flat 下使用
IVF_NPROBE          = 64          # 仅在 ivf_flat 下使用

# 输出命名
OUT_TAG_PREFIX  = "S_tag_topk"    # 会生成多分片：S_tag_topk_k{K}_part{idx}.parquet
OUT_TEXT_PREFIX = "S_text_topk"

# 预期输入（Step 6 产物）
IN_Z_TAG   = TMP_DIR / "Z_tag.parquet"
IN_Z_TEXT  = TMP_DIR / "Z_text.parquet"
IN_DOC     = TMP_DIR / "doc_clean.parquet"   # 仅用于校验行数与打印


In [8]:
# =========================
# Step 7 · Execute: 读取嵌入 → FAISS ANN → 分片落盘
# =========================
import os
import math
import numpy as np
import pandas as pd

# 1) 预检输入与目录
def _require(p: Path, desc: str):
    if not p.exists():
        raise FileNotFoundError(f"[FATAL] 缺少 {desc}: {p.as_posix()}")

for pth, desc in [(IN_Z_TAG, "Z_tag.parquet"),
                  (IN_Z_TEXT, "Z_text.parquet"),
                  (IN_DOC, "doc_clean.parquet")]:
    _require(pth, desc)

TMP_DIR.mkdir(parents=True, exist_ok=True)

# 2) 读取嵌入
def _load_Z(path: Path):
    df = pd.read_parquet(path, engine="fastparquet")
    # 只取 f0.. 列并按编号排序
    feat_cols = sorted([c for c in df.columns if c.startswith("f")], key=lambda x: int(x[1:]))
    X = df[feat_cols].to_numpy(dtype=np.float32, copy=True)
    # 再保险 L2 归一（训练已归一，这里确保数值稳定）
    n = np.linalg.norm(X, axis=1, keepdims=True)
    n[n < 1e-12] = 1.0
    X /= n
    return X

Z_tag  = _load_Z(IN_Z_TAG)
Z_text = _load_Z(IN_Z_TEXT)

doc_df = pd.read_parquet(IN_DOC, engine="fastparquet")
N = len(doc_df)
assert Z_tag.shape[0] == N and Z_text.shape[0] == N, \
    f"嵌入行数与文档数不一致: N={N}, Z_tag={Z_tag.shape}, Z_text={Z_text.shape}"

print(f"[INFO] Z_tag shape={Z_tag.shape}, Z_text shape={Z_text.shape}, docs={N}")

# 3) 构建/迁移 FAISS 索引
def _build_faiss_index(X, use_gpu=True, index_type="flat_ip"):
    import faiss

    d = X.shape[1]
    if index_type == "flat_ip":
        cpu_index = faiss.IndexFlatIP(d)
    elif index_type == "ivf_flat":
        quantizer = faiss.IndexFlatIP(d)
        cpu_index = faiss.IndexIVFFlat(quantizer, d, IVF_NLIST, faiss.METRIC_INNER_PRODUCT)
    else:
        raise ValueError("Unsupported FAISS index_type")

    # 迁移到 GPU（可用则多卡复制加速检索）
    index = cpu_index
    gpu_used = False
    if use_gpu:
        try:
            ngpu = faiss.get_num_gpus()
            if ngpu >= 1:
                # 复制到全部 GPU，默认 replicate 模式
                index = faiss.index_cpu_to_all_gpus(cpu_index, co=None)
                gpu_used = True
        except Exception as e:
            print("[WARN] GPU FAISS 不可用，退回 CPU：", e)

    # 训练 & 添加向量
    if isinstance(cpu_index, faiss.IndexIVFFlat):
        if not index.is_trained:
            # IVF 需训练量化器
            # 采样 200k 或全部中的一部分进行训练
            nsamp = min(200_000, X.shape[0])
            samp = X[np.random.choice(X.shape[0], nsamp, replace=False)]
            index.train(samp)
    index.add(X)  # 内存：平铺索引会把全量向量塞入

    return index, gpu_used

# 4) batched 检索 + 分片落盘
def _search_and_save(index, X, view_prefix: str):
    import faiss
    Kq = K_NEIGHBORS + 1  # 多取1个用于去自环

    total_edges = 0
    part_idx = 0
    buf_src, buf_dst, buf_sim = [], [], []

    def _flush_part():
        nonlocal total_edges, part_idx, buf_src, buf_dst, buf_sim
        if not buf_src: return
        df = pd.DataFrame({
            "src": np.asarray(buf_src, dtype=np.int64),
            "dst": np.asarray(buf_dst, dtype=np.int64),
            "sim": np.asarray(buf_sim, dtype=np.float32),
        })
        out = TMP_DIR / f"{view_prefix}_k{K_NEIGHBORS}_part{part_idx:04d}.parquet"
        df.to_parquet(out, engine="fastparquet", index=False)
        total_edges += len(df)
        print(f"[SAVE] {view_prefix} part#{part_idx} edges={len(df):,} -> {out.name}")
        part_idx += 1
        buf_src, buf_dst, buf_sim = [], [], []

    nq = X.shape[0]
    for s in range(0, nq, SEARCH_BATCH_Q):
        e = min(s + SEARCH_BATCH_Q, nq)
        xb = X[s:e]

        # FAISS 搜索
        D, I = index.search(xb, Kq)  # D: (b, Kq) sims, I: (b, Kq) idx

        # 去掉自环
        rows = np.arange(s, e, dtype=np.int64).reshape(-1, 1)
        mask_self = (I == rows)
        # 简单做法：把自环移除后再截取前 K
        new_dst = []
        new_sim = []
        for r in range(I.shape[0]):
            ids  = I[r][~mask_self[r]]
            sims = D[r][~mask_self[r]]
            if ids.shape[0] < K_NEIGHBORS:
                # 不足时用前若干顶上（极少发生）
                pad = K_NEIGHBORS - ids.shape[0]
                ids  = np.pad(ids,  (0, pad), constant_values=-1)
                sims = np.pad(sims, (0, pad), constant_values=0.0)
            else:
                ids  = ids[:K_NEIGHBORS]
                sims = sims[:K_NEIGHBORS]
            new_dst.append(ids)
            new_sim.append(sims)
        dst = np.vstack(new_dst).astype(np.int64)
        sim = np.vstack(new_sim).astype(np.float32)
        src = np.repeat(np.arange(s, e, dtype=np.int64).reshape(-1, 1), K_NEIGHBORS, axis=1)

        # 打平并缓冲
        buf_src.extend(src.reshape(-1).tolist())
        buf_dst.extend(dst.reshape(-1).tolist())
        buf_sim.extend(sim.reshape(-1).tolist())

        # 到达分片阈值就落盘
        if len(buf_src) >= SAVE_PART_EDGES:
            _flush_part()

        if (s // SEARCH_BATCH_Q) % 50 == 0:
            print(f"[PROG] {view_prefix}: processed {e:,}/{nq:,}")

    # flush 剩余
    _flush_part()

    # 写 manifest
    manifest = {
        "view": view_prefix,
        "k": K_NEIGHBORS,
        "index_type": FAISS_INDEX_TYPE,
        "gpu": True,
        "search_batch": SEARCH_BATCH_Q,
        "parts": sorted([p.name for p in TMP_DIR.glob(f"{view_prefix}_k{K_NEIGHBORS}_part*.parquet")]),
        "nodes": int(X.shape[0]),
        "dim": int(X.shape[1]),
        "total_edges": int(total_edges),
    }
    man_path = TMP_DIR / f"{view_prefix}_k{K_NEIGHBORS}_manifest.json"
    pd.Series(manifest).to_json(man_path)
    print(f"[MANIFEST] {man_path.name}  total_edges={total_edges:,}")

# ---- 执行：Tag ----
print("\n[BUILD] Tag view index ...")
tag_index, tag_gpu = _build_faiss_index(Z_tag, use_gpu=FAISS_USE_GPU, index_type=FAISS_INDEX_TYPE)
print(f"[READY] Tag index built. GPU={tag_gpu}. Start searching...")
_search_and_save(tag_index, Z_tag, OUT_TAG_PREFIX)

# ---- 执行：Text ----
print("\n[BUILD] Text view index ...")
text_index, text_gpu = _build_faiss_index(Z_text, use_gpu=FAISS_USE_GPU, index_type=FAISS_INDEX_TYPE)
print(f"[READY] Text index built. GPU={text_gpu}. Start searching...")
_search_and_save(text_index, Z_text, OUT_TEXT_PREFIX)

print("\n[Step 7] DONE: 已生成 Tag/Text 两视图的 top-K 相似边（分片 Parquet）与 manifest。")


[INFO] Z_tag shape=(521735, 256), Z_text shape=(521735, 256), docs=521735

[BUILD] Tag view index ...
[READY] Tag index built. GPU=False. Start searching...
[PROG] S_tag_topk: processed 8,192/521,735
[SAVE] S_tag_topk part#0 edges=2,048,000 -> S_tag_topk_k50_part0000.parquet
[SAVE] S_tag_topk part#1 edges=2,048,000 -> S_tag_topk_k50_part0001.parquet
[SAVE] S_tag_topk part#2 edges=2,048,000 -> S_tag_topk_k50_part0002.parquet
[SAVE] S_tag_topk part#3 edges=2,048,000 -> S_tag_topk_k50_part0003.parquet
[SAVE] S_tag_topk part#4 edges=2,048,000 -> S_tag_topk_k50_part0004.parquet
[SAVE] S_tag_topk part#5 edges=2,048,000 -> S_tag_topk_k50_part0005.parquet
[SAVE] S_tag_topk part#6 edges=2,048,000 -> S_tag_topk_k50_part0006.parquet
[SAVE] S_tag_topk part#7 edges=2,048,000 -> S_tag_topk_k50_part0007.parquet
[SAVE] S_tag_topk part#8 edges=2,048,000 -> S_tag_topk_k50_part0008.parquet
[SAVE] S_tag_topk part#9 edges=2,048,000 -> S_tag_topk_k50_part0009.parquet
[PROG] S_tag_topk: processed 417,792/521

In [9]:
# =========================
# Step 8 · Params: 图对称化 + 行归一化
# 仅使用 ./tmp 读写；自动读取 Step 7 的 manifest
# =========================
from pathlib import Path

TMP_DIR = Path("./tmp").resolve()

# 对称化策略：'max'（推荐，稳健且不放大小尺度差异），可改 'avg'
SYM_METHOD = "max"           # 'max' or 'avg'
ROW_NORM_EPS = 1e-12         # 行归一化的防零项

# 每个输出分片的最大边数（控制单文件大小）
SAVE_PART_EDGES = 2_000_000

# Step 7 产物前缀（不要改名）
TAG_PREFIX  = "S_tag_topk"
TEXT_PREFIX = "S_text_topk"

# 输出前缀（本步产物）
OUT_TAG_PREFIX  = "S_tag_symrow"   # 输出形如：S_tag_symrow_k{K}_partXXXX.parquet
OUT_TEXT_PREFIX = "S_text_symrow"

# 仅用于校验节点数
IN_DOC = TMP_DIR / "doc_clean.parquet"


In [10]:
# =========================
# Step 8 · Execute: 读取分片 → 构建稀疏 → 对称化 → 行归一化 → 分片落盘
# =========================
import json
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import sparse

def _require(p: Path, desc: str):
    if not p.exists():
        raise FileNotFoundError(f"[FATAL] 缺少 {desc}: {p.as_posix()}")

# 预检
_require(IN_DOC, "doc_clean.parquet")
doc_df = pd.read_parquet(IN_DOC, engine="fastparquet")
N = len(doc_df)

def _load_manifest(prefix: str):
    # 找到唯一 manifest
    mans = sorted(TMP_DIR.glob(f"{prefix}_k*_manifest.json"))
    if not mans:
        raise FileNotFoundError(f"[FATAL] 未找到 {prefix} 的 manifest")
    man_path = mans[0]
    man = json.loads(Path(man_path).read_text())
    K = int(man.get("k", 50))
    parts = man["parts"]
    print(f"[MAN] {man_path.name}  k={K}, parts={len(parts)}")
    return K, [TMP_DIR / p for p in parts]

def _build_sparse_from_parts(parts, n_nodes: int) -> sparse.coo_matrix:
    rows, cols, vals = [], [], []
    tot = 0
    for i, fp in enumerate(parts):
        df = pd.read_parquet(fp, engine="fastparquet")
        r = df["src"].to_numpy(np.int64, copy=False)
        c = df["dst"].to_numpy(np.int64, copy=False)
        v = df["sim"].to_numpy(np.float32, copy=False)
        rows.append(r); cols.append(c); vals.append(v)
        tot += len(df)
        if (i % 4) == 0:
            print(f"[LOAD] {i+1}/{len(parts)} parts, edges so far = {tot:,}")
        del df
    rows = np.concatenate(rows)
    cols = np.concatenate(cols)
    vals = np.concatenate(vals)
    print(f"[BUILD] COO ready: nnz={len(vals):,}")
    coo = sparse.coo_matrix((vals, (rows, cols)), shape=(n_nodes, n_nodes), dtype=np.float32)
    return coo

def _symmetrize(mat: sparse.coo_matrix, method: str) -> sparse.csr_matrix:
    A = mat.tocsr()
    AT = A.transpose().tocsr()
    if method == "max":
        # elementwise max
        A_sym = A.maximum(AT)
    elif method == "avg":
        A_sym = (A + AT).tocsr()
        A_sym.sum_duplicates()
        A_sym = A_sym.multiply(0.5)
    else:
        raise ValueError("SYM_METHOD must be 'max' or 'avg'")
    A_sym.eliminate_zeros()
    return A_sym

def _row_normalize(A: sparse.csr_matrix, eps: float = 1e-12) -> sparse.csr_matrix:
    rs = np.asarray(A.sum(axis=1)).ravel().astype(np.float64)
    rs[rs < eps] = 1.0
    inv = 1.0 / rs
    Dinv = sparse.diags(inv.astype(np.float32))
    P = Dinv @ A
    P.eliminate_zeros()
    return P.tocsr()

def _save_csr_parted(A: sparse.csr_matrix, out_prefix: str, k_val: int):
    # 以 COO 形式分片落盘
    C = A.tocoo()
    nnz = C.nnz
    print(f"[SAVE] {out_prefix}: nnz={nnz:,}  (可能 > N*K，因为对称化后边数会增加)")
    parts = []
    start = 0
    while start < nnz:
        end = min(start + SAVE_PART_EDGES, nnz)
        df = pd.DataFrame({
            "row": C.row[start:end].astype(np.int64),
            "col": C.col[start:end].astype(np.int64),
            "val": C.data[start:end].astype(np.float32),
        })
        outp = TMP_DIR / f"{out_prefix}_k{k_val}_part{start//SAVE_PART_EDGES:04d}.parquet"
        df.to_parquet(outp, engine="fastparquet", index=False)
        parts.append(outp.name)
        print(f"[SAVE] {outp.name} edges={len(df):,}")
        start = end
        del df
    # manifest
    manifest = {
        "view": out_prefix,
        "k_source": int(k_val),
        "sym_method": SYM_METHOD,
        "row_stochastic": True,
        "nodes": int(A.shape[0]),
        "nnz": int(nnz),
        "parts": parts,
    }
    man_path = TMP_DIR / f"{out_prefix}_k{k_val}_manifest.json"
    pd.Series(manifest).to_json(man_path)
    print(f"[MANIFEST] {man_path.name}  nnz={nnz:,}")

# ===== Tag view =====
print("\n[STEP8/TAG] 加载 manifest 与分片 ...")
k_tag, tag_parts = _load_manifest(TAG_PREFIX)
A_tag = _build_sparse_from_parts(tag_parts, N)
print("[STEP8/TAG] 对称化 ...")
A_tag_sym = _symmetrize(A_tag, SYM_METHOD)
print("[STEP8/TAG] 行归一化 ...")
P_tag = _row_normalize(A_tag_sym, ROW_NORM_EPS)
_save_csr_parted(P_tag, OUT_TAG_PREFIX, k_tag)

# ===== Text view =====
print("\n[STEP8/TEXT] 加载 manifest 与分片 ...")
k_text, text_parts = _load_manifest(TEXT_PREFIX)
A_text = _build_sparse_from_parts(text_parts, N)
print("[STEP8/TEXT] 对称化 ...")
A_text_sym = _symmetrize(A_text, SYM_METHOD)
print("[STEP8/TEXT] 行归一化 ...")
P_text = _row_normalize(A_text_sym, ROW_NORM_EPS)
_save_csr_parted(P_text, OUT_TEXT_PREFIX, k_text)

print("\n[Step 8] DONE: 已生成 Tag/Text 的对称且行归一的相似网络（分片 Parquet + manifest）。")



[STEP8/TAG] 加载 manifest 与分片 ...
[MAN] S_tag_topk_k50_manifest.json  k=50, parts=13
[LOAD] 1/13 parts, edges so far = 2,048,000
[LOAD] 5/13 parts, edges so far = 10,240,000
[LOAD] 9/13 parts, edges so far = 18,432,000
[LOAD] 13/13 parts, edges so far = 26,086,750
[BUILD] COO ready: nnz=26,086,750
[STEP8/TAG] 对称化 ...
[STEP8/TAG] 行归一化 ...
[SAVE] S_tag_symrow: nnz=28,159,756  (可能 > N*K，因为对称化后边数会增加)
[SAVE] S_tag_symrow_k50_part0000.parquet edges=2,000,000
[SAVE] S_tag_symrow_k50_part0001.parquet edges=2,000,000
[SAVE] S_tag_symrow_k50_part0002.parquet edges=2,000,000
[SAVE] S_tag_symrow_k50_part0003.parquet edges=2,000,000
[SAVE] S_tag_symrow_k50_part0004.parquet edges=2,000,000
[SAVE] S_tag_symrow_k50_part0005.parquet edges=2,000,000
[SAVE] S_tag_symrow_k50_part0006.parquet edges=2,000,000
[SAVE] S_tag_symrow_k50_part0007.parquet edges=2,000,000
[SAVE] S_tag_symrow_k50_part0008.parquet edges=2,000,000
[SAVE] S_tag_symrow_k50_part0009.parquet edges=2,000,000
[SAVE] S_tag_symrow_k50_part001

In [11]:
# =========================
# Step 9 · Params: 多视图相似网络快速融合（RA-SNF 风格，稀疏可扩展）
# 仅使用 ./tmp 读写；自动读取 Step 8 的 manifest
# =========================
from pathlib import Path

TMP_DIR = Path("./tmp").resolve()

# 输入前缀（不要改名）
TAG_SYM_PREFIX  = "S_tag_symrow"
TEXT_SYM_PREFIX = "S_text_symrow"

# 融合输出
FUSED_PREFIX    = "S_fused_symrow"   # 输出形如：S_fused_symrow_k{Kf}_partXXXX.parquet
K_FUSED         = 50                 # 融合后每行保留的近邻数
SAVE_PART_EDGES = 2_000_000          # 每个分片最大边数

# 视图的全局权重（在自适应权重基础上再乘以全局权重；通常保持 1.0）
W_TAG_GLOBAL  = 1.0
W_TEXT_GLOBAL = 1.0

# 行自适应的“尖锐度”度量：rho_i = sum_j p_ij^2 ∈ (0,1]，越大表示越自信
# 采用 归一化后 的行适应权重：alpha_tag_i = rho_tag_i / (rho_tag_i + rho_text_i + eps)
ROW_WEIGHT_EPS = 1e-12

# 仅用于校验节点数
IN_DOC = TMP_DIR / "doc_clean.parquet"


In [12]:
# =========================
# Step 9 · Execute: 稀疏自适应融合（无需稠密乘法，可扩展）
# =========================
import json
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import sparse

def _require(p: Path, desc: str):
    if not p.exists():
        raise FileNotFoundError(f"[FATAL] 缺少 {desc}: {p.as_posix()}")

# 预检
_require(IN_DOC, "doc_clean.parquet")
doc_df = pd.read_parquet(IN_DOC, engine="fastparquet")
N = len(doc_df)

def _load_manifest(prefix: str):
    mans = sorted(TMP_DIR.glob(f"{prefix}_k*_manifest.json"))
    if not mans:
        raise FileNotFoundError(f"[FATAL] 未找到 {prefix} 的 manifest")
    man_path = mans[0]
    man = json.loads(Path(man_path).read_text())
    k = int(man.get("k_source", man.get("k", 50)))
    parts = [TMP_DIR / p for p in man["parts"]]
    print(f"[MAN] {man_path.name}  k={k}, parts={len(parts)}")
    return k, parts

def _build_csr(parts, n_nodes: int) -> sparse.csr_matrix:
    rows, cols, vals = [], [], []
    tot = 0
    for i, fp in enumerate(parts):
        df = pd.read_parquet(fp, engine="fastparquet")
        rows.append(df["row"].to_numpy(np.int64, copy=False))
        cols.append(df["col"].to_numpy(np.int64, copy=False))
        vals.append(df["val"].to_numpy(np.float32, copy=False))
        tot += len(df)
        if (i % 4) == 0:
            print(f"[LOAD] {i+1}/{len(parts)} parts, edges so far = {tot:,}")
        del df
    r = np.concatenate(rows); c = np.concatenate(cols); v = np.concatenate(vals)
    A = sparse.coo_matrix((v, (r, c)), shape=(n_nodes, n_nodes), dtype=np.float32).tocsr()
    A.sum_duplicates(); A.eliminate_zeros()
    print(f"[CSR] nnz={A.nnz:,}")
    return A

# 读取两视图的行随机矩阵（Step 8 输出）
print("\n[FUSE] 加载 Tag/Text 行随机图 ...")
k_tag,  tag_parts  = _load_manifest(TAG_SYM_PREFIX)
k_text, text_parts = _load_manifest(TEXT_SYM_PREFIX)
P_tag  = _build_csr(tag_parts,  N)
P_text = _build_csr(text_parts, N)

# 计算每行尖锐度 rho = sum p^2（可向量化：对 data^2 按行聚合）
def row_concentration_squared_sum(P: sparse.csr_matrix) -> np.ndarray:
    d2 = np.square(P.data, dtype=np.float32)
    # reduceat 以行边界 indptr 聚合
    return np.add.reduceat(d2, P.indptr[:-1])

rho_tag  = row_concentration_squared_sum(P_tag).astype(np.float64)
rho_text = row_concentration_squared_sum(P_text).astype(np.float64)
den = rho_tag + rho_text + ROW_WEIGHT_EPS
alpha_tag  = (rho_tag  / den) * W_TAG_GLOBAL
alpha_text = (rho_text / den) * W_TEXT_GLOBAL
# 归一化（以防全局权重不等）
den2 = alpha_tag + alpha_text + ROW_WEIGHT_EPS
alpha_tag  = alpha_tag  / den2
alpha_text = alpha_text / den2

print("[FUSE] 行自适应权重统计：")
for name, a in [("tag", alpha_tag), ("text", alpha_text)]:
    print(f"  - alpha_{name}: min={a.min():.4f}, median={np.median(a):.4f}, max={a.max():.4f}")

# 按行缩放（不构造对角阵，直接重复权重到 data）
def scale_rows_csr(P: sparse.csr_matrix, alpha: np.ndarray) -> sparse.csr_matrix:
    P = P.tocsr(copy=True)
    deg = np.diff(P.indptr)
    rep = np.repeat(alpha, deg)
    P.data = P.data * rep.astype(np.float32)
    return P

print("[FUSE] 按行自适应权重缩放并稀疏求和 ...")
Pt = scale_rows_csr(P_tag,  alpha_tag)
Px = scale_rows_csr(P_text, alpha_text)
P_sum = (Pt + Px).tocsr()
P_sum.sum_duplicates(); P_sum.eliminate_zeros()
print(f"[FUSE] 融合后（未裁剪） nnz={P_sum.nnz:,}")

# 行内 top-K_fused，保持稀疏度
def csr_row_topk(P: sparse.csr_matrix, k: int) -> sparse.csr_matrix:
    indptr, indices, data = P.indptr, P.indices, P.data
    N = P.shape[0]
    deg = np.diff(indptr)
    k_keep = np.minimum(k, deg)

    nnz_keep = int(k_keep.sum())
    new_indptr = np.zeros(N + 1, dtype=np.int64)
    new_indptr[1:] = np.cumsum(k_keep)

    new_indices = np.empty(nnz_keep, dtype=np.int64)
    new_data    = np.empty(nnz_keep, dtype=np.float32)

    pos = 0
    for i in range(N):
        s, e = indptr[i], indptr[i+1]
        kk = k_keep[i]
        if kk == 0: continue
        row_idx = indices[s:e]; row_val = data[s:e]
        if e - s <= k:
            # 直接保留
            new_indices[pos:pos+kk] = row_idx
            new_data[pos:pos+kk]    = row_val
        else:
            part = np.argpartition(row_val, -kk)[-kk:]
            ords = np.argsort(-row_val[part])
            sel  = part[ords]
            new_indices[pos:pos+kk] = row_idx[sel]
            new_data[pos:pos+kk]    = row_val[sel]
        pos += kk

    out = sparse.csr_matrix((new_data, new_indices, new_indptr), shape=P.shape, dtype=np.float32)
    out.eliminate_zeros()
    return out

print(f"[FUSE] 行内保留 top-{K_FUSED} ...")
P_topk = csr_row_topk(P_sum, K_FUSED)

# 行归一（再保险）
def row_normalize(P: sparse.csr_matrix, eps: float = 1e-12) -> sparse.csr_matrix:
    rs = np.asarray(P.sum(axis=1)).ravel().astype(np.float64)
    rs[rs < eps] = 1.0
    inv = (1.0 / rs).astype(np.float32)
    Dinv = sparse.diags(inv)
    Q = Dinv @ P
    Q.eliminate_zeros()
    return Q.tocsr()

P_fused = row_normalize(P_topk)

print(f"[FUSE] 最终 nnz={P_fused.nnz:,}  （理论上 ≤ N * K_fused 的 2~3 倍以内）")

# 分片落盘
def save_csr_parted(A: sparse.csr_matrix, out_prefix: str, k_val: int):
    C = A.tocoo()
    nnz = C.nnz
    print(f"[SAVE] {out_prefix}: nnz={nnz:,}")
    parts = []
    start = 0
    while start < nnz:
        end = min(start + SAVE_PART_EDGES, nnz)
        df = pd.DataFrame({
            "row": C.row[start:end].astype(np.int64),
            "col": C.col[start:end].astype(np.int64),
            "val": C.data[start:end].astype(np.float32),
        })
        outp = TMP_DIR / f"{out_prefix}_k{k_val}_part{start//SAVE_PART_EDGES:04d}.parquet"
        df.to_parquet(outp, engine="fastparquet", index=False)
        parts.append(outp.name)
        print(f"[SAVE] {outp.name} edges={len(df):,}")
        start = end
        del df
    manifest = {
        "view": out_prefix,
        "k_target": int(k_val),
        "nodes": int(A.shape[0]),
        "nnz": int(nnz),
        "parts": parts,
        "alpha_row_stat": {
            "tag":  {"min": float(alpha_tag.min()),  "median": float(np.median(alpha_tag)),  "max": float(alpha_tag.max())},
            "text": {"min": float(alpha_text.min()), "median": float(np.median(alpha_text)), "max": float(alpha_text.max())},
        },
        "w_global": {"tag": float(W_TAG_GLOBAL), "text": float(W_TEXT_GLOBAL)}
    }
    man_path = TMP_DIR / f"{out_prefix}_k{k_val}_manifest.json"
    pd.Series(manifest).to_json(man_path)
    print(f"[MANIFEST] {man_path.name}  nnz={nnz:,}")

save_csr_parted(P_fused, FUSED_PREFIX, K_FUSED)
print("\n[Step 9] DONE: 已生成融合后的行随机相似网络（分片 Parquet + manifest）。")



[FUSE] 加载 Tag/Text 行随机图 ...
[MAN] S_tag_symrow_k50_manifest.json  k=50, parts=15
[MAN] S_text_symrow_k50_manifest.json  k=50, parts=15
[LOAD] 1/15 parts, edges so far = 2,000,000
[LOAD] 5/15 parts, edges so far = 10,000,000
[LOAD] 9/15 parts, edges so far = 18,000,000
[LOAD] 13/15 parts, edges so far = 26,000,000
[CSR] nnz=28,159,756
[LOAD] 1/15 parts, edges so far = 2,000,000
[LOAD] 5/15 parts, edges so far = 10,000,000
[LOAD] 9/15 parts, edges so far = 18,000,000
[LOAD] 13/15 parts, edges so far = 26,000,000
[CSR] nnz=28,161,106
[FUSE] 行自适应权重统计：
  - alpha_tag: min=0.3524, median=0.5000, max=0.6552
  - alpha_text: min=0.3448, median=0.5000, max=0.6476
[FUSE] 按行自适应权重缩放并稀疏求和 ...
[FUSE] 融合后（未裁剪） nnz=56,317,938
[FUSE] 行内保留 top-50 ...
[FUSE] 最终 nnz=26,086,750  （理论上 ≤ N * K_fused 的 2~3 倍以内）
[SAVE] S_fused_symrow: nnz=26,086,750
[SAVE] S_fused_symrow_k50_part0000.parquet edges=2,000,000
[SAVE] S_fused_symrow_k50_part0001.parquet edges=2,000,000
[SAVE] S_fused_symrow_k50_part0002.parquet edg

In [13]:
# 行为视图 · Step 1：输出列名（只读表头，零开销）
from pathlib import Path
import pandas as pd

csv_path = Path("data/metadata_merged.csv").resolve()
assert csv_path.exists(), f"[FATAL] 文件不存在：{csv_path}"

# 只读表头（nrows=0），避免加载大文件
df_head = pd.read_csv(csv_path, nrows=0)

cols = list(df_head.columns)
print(f"[INFO] 文件路径: {csv_path}")
print(f"[INFO] 列名（共 {len(cols)} 列）:")
print(cols)


[INFO] 文件路径: /home/koyo/workspace/recsys/data/metadata_merged.csv
[INFO] 列名（共 31 列）:
['Id', 'Title', 'Slug', 'Tags', 'CreatorUserId', 'OwnerUserId', 'OwnerOrganizationId', 'CurrentDatasetVersionId', 'CurrentDatasourceVersionId', 'ForumId', 'Type', 'CreationDate', 'LastActivityDate', 'TotalViews', 'TotalDownloads', 'TotalVotes', 'TotalKernels', 'Medal', 'MedalAwardDate', 'Subtitle', 'Description', 'CreationDate_dt', 'LastActivityDate_dt', 'age_days', 'days_since_last_activity', 'active_30d', 'has_tags', 'TotalViews_log1p', 'TotalDownloads_log1p', 'TotalVotes_log1p', 'TotalKernels_log1p']


In [14]:
# =========================
# Step B1 · Params: 行为视图——对齐与基表准备
# =========================
from pathlib import Path

TMP_DIR = Path("./tmp").resolve()
CSV_PATH = Path("data/metadata_merged.csv").resolve()

# 需要的列（行为信号）
BEH_DISCRETE_COLS = [
    "CreatorUserId", "OwnerUserId", "OwnerOrganizationId", "ForumId"
]
BEH_CONT_COLS = [
    "TotalViews_log1p", "TotalDownloads_log1p", "TotalVotes_log1p", "TotalKernels_log1p",
    "age_days", "days_since_last_activity", "active_30d", "Medal"
]

# 只读下述列，避免加载无关内容
USECOLS = ["Id"] + BEH_DISCRETE_COLS + BEH_CONT_COLS

# 读取 CSV 的 dtype（pandas 可空整数 Int64 允许 NaN）
CSV_DTYPES = {
    "Id": "Int64",
    "CreatorUserId": "Int64",
    "OwnerUserId": "Int64",
    "OwnerOrganizationId": "Int64",
    "ForumId": "Int64",
    "TotalViews_log1p": "float32",
    "TotalDownloads_log1p": "float32",
    "TotalVotes_log1p": "float32",
    "TotalKernels_log1p": "float32",
    "age_days": "Int64",
    "days_since_last_activity": "Int64",
    "active_30d": "boolean",
    "Medal": "float32",
}

# 输出文件
OUT_BEH_BASE = TMP_DIR / "beh_base.parquet"


In [15]:
# =========================
# Step B1 · Execute
# =========================
import pandas as pd
import numpy as np

# 0) 基础检查
assert CSV_PATH.exists(), f"[FATAL] 不存在: {CSV_PATH}"
TMP_DIR.mkdir(parents=True, exist_ok=True)

# 1) 读取 doc 映射（优先 index_map.parquet；否则从 doc_clean 取）
idx_map_path = TMP_DIR / "index_map.parquet"
doc_clean_path = TMP_DIR / "doc_clean.parquet"
if idx_map_path.exists():
    idmap = pd.read_parquet(idx_map_path, engine="fastparquet")
    # 期望列: doc_idx, Id
    assert {"doc_idx", "Id"} <= set(idmap.columns), "[FATAL] index_map.parquet 缺少列"
    idmap = idmap[["doc_idx", "Id"]].copy()
else:
    # 退化：从 doc_clean 提取 doc_idx, Id
    assert doc_clean_path.exists(), "[FATAL] 缺少 index_map.parquet 与 doc_clean.parquet"
    dclean = pd.read_parquet(doc_clean_path, engine="fastparquet")
    assert {"doc_idx", "Id"} <= set(dclean.columns), "[FATAL] doc_clean.parquet 缺少 doc_idx/Id"
    idmap = dclean[["doc_idx", "Id"]].copy()
N = len(idmap)
print(f"[INFO] 映射加载完成: N={N}")

# 2) 仅读取需要的列（零拷贝列名级读取）
df = pd.read_csv(
    CSV_PATH, usecols=USECOLS, dtype=CSV_DTYPES, low_memory=False, memory_map=True
)
print(f"[INFO] CSV 读取完成: shape={df.shape}")

# 3) 对齐（只保留在 idmap 中的 Id；并按 doc_idx 排序）
df = df.merge(idmap, on="Id", how="right", validate="one_to_one")
df = df.sort_values("doc_idx").reset_index(drop=True)

# 4) 断言行数与索引一致（确保与其他视图对齐）
assert len(df) == N, f"[FATAL] 行数不一致: beh={len(df)}, N={N}"
assert (df["doc_idx"].to_numpy() == np.arange(N)).all(), "[FATAL] doc_idx 未按 0..N-1 对齐"

# 5) 缺失处理
# 离散 ID: 缺失记为 -1（后续共现时跳过 -1）
for c in ["CreatorUserId", "OwnerUserId", "OwnerOrganizationId", "ForumId"]:
    if c in df.columns:
        df[c] = df[c].astype("Int64")
        df[c] = df[c].fillna(-1).astype("Int64")

# 连续列: 缺失补 0；活跃布尔缺失补 False
for c in ["TotalViews_log1p","TotalDownloads_log1p","TotalVotes_log1p","TotalKernels_log1p","Medal"]:
    if c in df.columns:
        df[c] = df[c].astype("float32").fillna(0.0)
for c in ["age_days", "days_since_last_activity"]:
    if c in df.columns:
        df[c] = df[c].astype("Int64").fillna(0)
if "active_30d" in df.columns:
    df["active_30d"] = df["active_30d"].astype("boolean").fillna(False)

# 6) 仅保留需要列，按 doc_idx 排列
keep_cols = ["doc_idx","Id"] + BEH_DISCRETE_COLS + BEH_CONT_COLS
beh_base = df[keep_cols].copy()

# 7) 保存到 ./tmp（fastparquet）
beh_base.to_parquet(OUT_BEH_BASE, engine="fastparquet", index=False)

# 8) 打印快速检查
def _uniq(c): 
    s = beh_base[c]
    return int(s[s != -1].nunique()) if pd.api.types.is_integer_dtype(s.dtype) else int(s.nunique())

print("[OK] 行为基表已生成:", OUT_BEH_BASE)
print("[STATS] 非缺失唯一计数（离散）:")
for c in BEH_DISCRETE_COLS:
    print(f"  - {c}: uniq={_uniq(c)}")
print("[STATS] 连续列（均值±std，最小/最大）:")
for c in ["TotalViews_log1p","TotalDownloads_log1p","TotalVotes_log1p","TotalKernels_log1p"]:
    arr = beh_base[c].to_numpy(dtype=np.float32, copy=False)
    print(f"  - {c}: mean={arr.mean():.3f}, std={arr.std():.3f}, min={arr.min():.3f}, max={arr.max():.3f}")
print("[STATS] 时效列样例:")
if "age_days" in beh_base.columns:
    a = beh_base["age_days"].astype("int64").to_numpy()
    print(f"  - age_days: min={a.min()}, median={np.median(a)}, max={a.max()}")
if "days_since_last_activity" in beh_base.columns:
    a = beh_base["days_since_last_activity"].astype("int64").to_numpy()
    print(f"  - days_since_last_activity: min={a.min()}, median={np.median(a)}, max={a.max()}")
if "active_30d" in beh_base.columns:
    rate = float(beh_base["active_30d"].mean())
    print(f"  - active_30d: true_rate={rate:.3f}")


[INFO] 映射加载完成: N=521735
[INFO] CSV 读取完成: shape=(521735, 13)
[OK] 行为基表已生成: /home/koyo/workspace/recsys/tmp/beh_base.parquet
[STATS] 非缺失唯一计数（离散）:
  - CreatorUserId: uniq=192013
  - OwnerUserId: uniq=191796
  - OwnerOrganizationId: uniq=390
  - ForumId: uniq=521735
[STATS] 连续列（均值±std，最小/最大）:
  - TotalViews_log1p: mean=5.164, std=2.018, min=0.000, max=16.299
  - TotalDownloads_log1p: mean=2.578, std=1.789, min=0.000, max=13.760
  - TotalVotes_log1p: mean=0.594, std=0.962, min=0.000, max=10.895
  - TotalKernels_log1p: mean=0.359, std=0.640, min=0.000, max=8.921
[STATS] 时效列样例:
  - age_days: min=78, median=728.0, max=3762
  - days_since_last_activity: min=79, median=729.0, max=3023
  - active_30d: true_rate=0.000


In [16]:
# =========================
# Step B2 · Params: 协同/归属相似图 S_ids
# =========================
from pathlib import Path

TMP_DIR = Path("./tmp").resolve()

# 输入
IN_BEH_BASE = TMP_DIR / "beh_base.parquet"

# 使用的离散字段（不含 ForumId）
IDS_FIELDS = ["CreatorUserId", "OwnerUserId", "OwnerOrganizationId"]

# 字段全局权重（相对强度）
W_FIELD = {
    "CreatorUserId": 1.00,
    "OwnerUserId":  0.80,
    "OwnerOrganizationId": 0.60,
}

# 大组惩罚（IDF 风格）：weight *= 1 / log1p(group_size)
# 对超大组直接跳过（避免爆炸）
MAX_GROUP_SIZE = {
    "CreatorUserId": 5000,
    "OwnerUserId":  5000,
    "OwnerOrganizationId": 20000,
}

# 每行保留的近邻数
K_IDS = 50

# 批处理大小（按 doc_idx 分块处理）
DOC_BATCH = 50_000

# 保存分片的边数上限（控制单文件大小）
SAVE_PART_EDGES = 2_000_000

# 输出前缀
OUT_PREFIX_IDS = "S_ids_topk"


In [17]:
# =========================
# Step B2 · Execute
# =========================
import numpy as np
import pandas as pd

def _require(p: Path, desc: str):
    if not p.exists():
        raise FileNotFoundError(f"[FATAL] 缺少 {desc}: {p.as_posix()}")

_require(IN_BEH_BASE, "beh_base.parquet")

# 1) 读入基表（仅需要的列，且保证 doc_idx 顺序）
base = pd.read_parquet(IN_BEH_BASE, engine="fastparquet")
assert "doc_idx" in base.columns and (base["doc_idx"].to_numpy() == np.arange(len(base))).all()
N = len(base)

# 2) 为每个字段构建“值→成员doc列表”的**索引（稀疏）**
#    使用排序数组 + unique 边界，而非大字典，节省内存
group_indices = {}   # field -> dict with arrays: uniq_vals, start, end, doc_order
for f in IDS_FIELDS:
    vals = base[f].to_numpy(copy=False)
    # 去掉 -1（缺失）
    mask = vals != -1
    doc_ids = np.arange(N, dtype=np.int64)[mask]
    vals_nz = vals[mask].astype(np.int64, copy=False)

    # 按值排序，构建 unique 边界
    order = np.argsort(vals_nz, kind="mergesort")
    vals_sorted = vals_nz[order]
    docs_sorted = doc_ids[order]
    uniq_vals, start_idx = np.unique(vals_sorted, return_index=True)
    # 计算 end 边界
    end_idx = np.empty_like(start_idx)
    end_idx[:-1] = start_idx[1:]
    end_idx[-1] = len(vals_sorted)

    group_indices[f] = {
        "uniq_vals": uniq_vals,          # 唯一取值（不含 -1）
        "start": start_idx,              # 在 docs_sorted 中的起始
        "end": end_idx,                  # 在 docs_sorted 中的结束
        "docs_sorted": docs_sorted,      # 与 uniq_vals 对应的拼接成员
    }
    print(f"[INDEX] {f}: groups={len(uniq_vals)} "
          f"(avg size ≈ {len(docs_sorted)/max(1,len(uniq_vals)):.2f})")

# 3) 行处理：逐文档聚合候选邻居（来自 3 个字段的同组），做 IDF 抑制 & 字段权重
def get_group_members(field, val):
    """返回某字段某值的成员 doc 列表（np.ndarray[int]），若无或超限则返回 None。"""
    if val == -1:
        return None
    gi = group_indices[field]
    uniq = gi["uniq_vals"]
    pos = np.searchsorted(uniq, val)
    if pos >= len(uniq) or uniq[pos] != val:
        return None
    s, e = gi["start"][pos], gi["end"][pos]
    size = e - s
    # 大组截断：直接跳过，避免 O(g^2) 膨胀
    if size > MAX_GROUP_SIZE[field]:
        return None
    return gi["docs_sorted"][s:e]

def process_chunk(s, e):
    rows, cols, vals = [], [], []
    for i in range(s, e):
        cand_ids = []
        cand_w   = []

        # 三个字段的组成员
        for f in IDS_FIELDS:
            val = int(base.at[i, f])
            mem = get_group_members(f, val)
            if mem is None:
                continue
            # 移除自身
            if mem.size and (mem[0] <= i <= mem[-1]):
                # 快速自检：仅在可能时做布尔掩码
                mem = mem[mem != i]
            if mem.size == 0:
                continue
            gsize = mem.size
            # 权重：字段权重 × IDF 抑制
            w = (W_FIELD[f] / np.log1p(gsize)).astype(np.float32)
            cand_ids.append(mem)
            # 广播同长权重
            cand_w.append(np.full(gsize, w, dtype=np.float32))

        if not cand_ids:
            continue

        cand_ids = np.concatenate(cand_ids)
        cand_w   = np.concatenate(cand_w)

        # 合并重复候选（同一邻居可来自不同字段）：按 id 聚合求和
        uniq_nbrs, inv = np.unique(cand_ids, return_inverse=True)
        w_sum = np.bincount(inv, weights=cand_w).astype(np.float32)

        # 取 top-K
        k = min(K_IDS, uniq_nbrs.size)
        if k == 0:
            continue
        if uniq_nbrs.size > k:
            part = np.argpartition(w_sum, -k)[-k:]
            ords = np.argsort(-w_sum[part])
            sel = part[ords]
            nbrs = uniq_nbrs[sel]
            wsel = w_sum[sel]
        else:
            nbrs = uniq_nbrs
            wsel = w_sum

        # 行归一
        ssum = float(wsel.sum())
        if ssum <= 0:
            continue
        wsel = (wsel / ssum).astype(np.float32)

        # 写入缓冲
        rows.append(np.full(k, i, dtype=np.int64))
        cols.append(nbrs.astype(np.int64))
        vals.append(wsel)

    if rows:
        return np.concatenate(rows), np.concatenate(cols), np.concatenate(vals)
    else:
        return np.empty(0, dtype=np.int64), np.empty(0, dtype=np.int64), np.empty(0, dtype=np.float32)

# 4) 批处理 + 分片落盘
part_idx = 0
buf_r, buf_c, buf_v = [], [], []
total_edges = 0

def flush_part():
    global part_idx, total_edges, buf_r, buf_c, buf_v
    if not buf_r:
        return
    r = np.concatenate(buf_r); c = np.concatenate(buf_c); v = np.concatenate(buf_v)
    df_out = pd.DataFrame({"row": r, "col": c, "val": v})
    outp = TMP_DIR / f"{OUT_PREFIX_IDS}_k{K_IDS}_part{part_idx:04d}.parquet"
    df_out.to_parquet(outp, engine="fastparquet", index=False)
    total_edges += len(df_out)
    print(f"[SAVE] part#{part_idx} edges={len(df_out):,} -> {outp.name}")
    part_idx += 1
    buf_r, buf_c, buf_v = [], [], []

for s in range(0, N, DOC_BATCH):
    e = min(s + DOC_BATCH, N)
    r, c, v = process_chunk(s, e)
    if r.size > 0:
        buf_r.append(r); buf_c.append(c); buf_v.append(v)
    # 控制单文件大小
    cur = sum(map(len, buf_r))
    if cur >= SAVE_PART_EDGES:
        flush_part()
    if (s // DOC_BATCH) % 2 == 0:
        print(f"[PROG] processed {e:,}/{N:,}")

flush_part()

# 5) 写 manifest
man = {
    "view": OUT_PREFIX_IDS,
    "k": int(K_IDS),
    "nodes": int(N),
    "parts": sorted([p.name for p in TMP_DIR.glob(f"{OUT_PREFIX_IDS}_k{K_IDS}_part*.parquet")]),
    "fields": IDS_FIELDS,
    "weights": W_FIELD,
    "max_group_size": MAX_GROUP_SIZE,
    "note": "row-stochastic; no self-loops; IDF-like group penalty applied; ForumId skipped"
}
man_path = TMP_DIR / f"{OUT_PREFIX_IDS}_k{K_IDS}_manifest.json"
pd.Series(man).to_json(man_path)
print(f"[MANIFEST] {man_path.name}  total_edges={total_edges:,}")

print("\n[Step B2] DONE: 已生成协同/归属相似图 S_ids（分片 Parquet + manifest）。")


[INDEX] CreatorUserId: groups=192013 (avg size ≈ 2.72)
[INDEX] OwnerUserId: groups=191796 (avg size ≈ 2.71)
[INDEX] OwnerOrganizationId: groups=390 (avg size ≈ 6.61)
[PROG] processed 50,000/521,735
[PROG] processed 150,000/521,735
[SAVE] part#0 edges=2,535,282 -> S_ids_topk_k50_part0000.parquet
[PROG] processed 250,000/521,735
[PROG] processed 350,000/521,735
[SAVE] part#1 edges=2,348,909 -> S_ids_topk_k50_part0001.parquet
[PROG] processed 450,000/521,735
[PROG] processed 521,735/521,735
[SAVE] part#2 edges=1,278,456 -> S_ids_topk_k50_part0002.parquet
[MANIFEST] S_ids_topk_k50_manifest.json  total_edges=6,162,647

[Step B2] DONE: 已生成协同/归属相似图 S_ids（分片 Parquet + manifest）。


In [18]:
# =========================
# Step B3 · Params: 参与度相似图 S_eng（FAISS 余弦 top-K）
# =========================
from pathlib import Path

TMP_DIR = Path("./tmp").resolve()
IN_BEH_BASE = TMP_DIR / "beh_base.parquet"

# 使用的连续参与度特征
ENG_COLS = ["TotalViews_log1p", "TotalDownloads_log1p", "TotalVotes_log1p", "TotalKernels_log1p"]

# 时衰修正（温和）：E *= exp(-lambda_age*age_years) * exp(-lambda_idle*idle_years) * (1 + gamma_active*active_30d)
LAMBDA_AGE   = 0.10   # 对 age_days 的衰减系数（/年）
LAMBDA_IDLE  = 0.10   # 对 days_since_last_activity 的衰减系数（/年）
GAMMA_ACTIVE = 0.10   # active_30d 的加成（0 或 0.1）

# ANN 检索参数
K_ENG             = 50
SEARCH_BATCH_Q    = 8192
SAVE_PART_EDGES   = 2_000_000

# FAISS 配置
FAISS_USE_GPU     = True
FAISS_INDEX_TYPE  = "flat_ip"     # 参与度维度很小，Flat 精确检索足够快
# 如需改为 IVF：FAISS_INDEX_TYPE="ivf_flat" 并设置：
IVF_NLIST         = 4096
IVF_NPROBE        = 64

OUT_PREFIX_ENG    = "S_eng_topk"


In [19]:
# =========================
# Step B3 · Execute
# =========================
import os
import math
import numpy as np
import pandas as pd

def _require(p: Path, desc: str):
    if not p.exists():
        raise FileNotFoundError(f"[FATAL] 缺少 {desc}: {p.as_posix()}")

_require(IN_BEH_BASE, "beh_base.parquet")
base = pd.read_parquet(IN_BEH_BASE, engine="fastparquet")
N = len(base)
assert (base["doc_idx"].to_numpy() == np.arange(N)).all(), "[FATAL] beh_base 未与 doc_idx 对齐"

# 1) 构造参与度向量（N×4）
X = base[ENG_COLS].to_numpy(dtype=np.float32, copy=True)

# 2) 温和时衰修正
age_years  = base["age_days"].to_numpy(dtype=np.float32, copy=False) / 365.0
idle_years = base["days_since_last_activity"].to_numpy(dtype=np.float32, copy=False) / 365.0
active     = base["active_30d"].astype(np.float32).to_numpy(copy=False)

decay = np.exp(-LAMBDA_AGE * age_years) * np.exp(-LAMBDA_IDLE * idle_years) * (1.0 + GAMMA_ACTIVE * active)
decay = decay.reshape(-1, 1).astype(np.float32)
X *= decay

# 3) 行 L2 归一（余弦=内积），并标记全零行
row_norm = np.linalg.norm(X, axis=1, keepdims=True)
mask_valid = (row_norm.squeeze(1) > 0)
X_valid = X[mask_valid].copy()
row_norm_valid = np.linalg.norm(X_valid, axis=1, keepdims=True)
X_valid /= np.maximum(row_norm_valid, 1e-12)

valid_doc_idx = np.flatnonzero(mask_valid).astype(np.int64)
n_valid = int(mask_valid.sum())
n_invalid = int((~mask_valid).sum())
print(f"[ENG] valid rows={n_valid:,}, invalid(all-zero)={n_invalid:,}")

# 4) 构建 FAISS 索引（GPU 优先）
def _build_faiss_index(X, use_gpu=True, index_type="flat_ip"):
    import faiss
    d = X.shape[1]
    if index_type == "flat_ip":
        cpu_index = faiss.IndexFlatIP(d)
    elif index_type == "ivf_flat":
        quantizer = faiss.IndexFlatIP(d)
        cpu_index = faiss.IndexIVFFlat(quantizer, d, IVF_NLIST, faiss.METRIC_INNER_PRODUCT)
    else:
        raise ValueError("Unsupported FAISS index_type")

    index = cpu_index
    gpu_used = False
    if use_gpu:
        try:
            ngpu = faiss.get_num_gpus()
            if ngpu >= 1:
                index = faiss.index_cpu_to_all_gpus(cpu_index, co=None)
                gpu_used = True
        except Exception as e:
            print("[WARN] GPU FAISS 不可用，退回 CPU：", e)

    if isinstance(cpu_index, faiss.IndexIVFFlat):
        if not index.is_trained:
            nsamp = min(200_000, X.shape[0])
            samp = X[np.random.choice(X.shape[0], nsamp, replace=False)]
            index.train(samp)

    index.add(X)
    return index, gpu_used

print("[BUILD] FAISS index for engagement ...")
index, gpu_used = _build_faiss_index(X_valid, use_gpu=FAISS_USE_GPU, index_type=FAISS_INDEX_TYPE)
print(f"[READY] index built. GPU={gpu_used}. Start searching...")

# 5) batched 检索（只对 valid 行），K+1 去自环
Kq = K_ENG + 1
buf_src, buf_dst, buf_sim = [], [], []
total_edges = 0
part_idx = 0

def _flush():
    global part_idx, total_edges, buf_src, buf_dst, buf_sim
    if not buf_src:
        return
    r = np.concatenate(buf_src); c = np.concatenate(buf_dst); v = np.concatenate(buf_sim)
    df = pd.DataFrame({"row": r, "col": c, "val": v})
    outp = TMP_DIR / f"{OUT_PREFIX_ENG}_k{K_ENG}_part{part_idx:04d}.parquet"
    df.to_parquet(outp, engine="fastparquet", index=False)
    total_edges += len(df)
    print(f"[SAVE] part#{part_idx} edges={len(df):,} -> {outp.name}")
    part_idx += 1
    buf_src, buf_dst, buf_sim = [], [], []

nq = X_valid.shape[0]
for s in range(0, nq, SEARCH_BATCH_Q):
    e = min(s + SEARCH_BATCH_Q, nq)
    xb = X_valid[s:e]
    D, I = index.search(xb, Kq)  # (b, Kq)

    # 将 I（valid 子集索引）映射回全局 doc_idx
    rows_global = valid_doc_idx[s:e]
    dst_global = valid_doc_idx[I]

    # 去自环并截取 K
    new_rows, new_cols, new_vals = [], [], []
    for r in range(e - s):
        rg = rows_global[r]
        ids  = dst_global[r]
        sims = D[r]
        keep = ids != rg
        ids  = ids[keep]
        sims = sims[keep]
        if ids.size == 0:
            continue
        if ids.size > K_ENG:
            ids  = ids[:K_ENG]
            sims = sims[:K_ENG]
        # 行归一（避免极小负数，截断到 >=0）
        sims = np.clip(sims, 0.0, None)
        ssum = float(sims.sum())
        if ssum <= 0:
            continue
        sims = (sims / ssum).astype(np.float32)

        new_rows.append(np.full(ids.size, rg, dtype=np.int64))
        new_cols.append(ids.astype(np.int64))
        new_vals.append(sims)

    if new_rows:
        buf_src.append(np.concatenate(new_rows))
        buf_dst.append(np.concatenate(new_cols))
        buf_sim.append(np.concatenate(new_vals))

    # 分片
    cur = sum(map(len, buf_src))
    if cur >= SAVE_PART_EDGES:
        _flush()

    if (s // SEARCH_BATCH_Q) % 50 == 0:
        print(f"[PROG] {OUT_PREFIX_ENG}: processed {e:,}/{nq:,}")

_flush()

# 6) 写 manifest
parts = sorted([p.name for p in TMP_DIR.glob(f"{OUT_PREFIX_ENG}_k{K_ENG}_part*.parquet")])
man = {
    "view": OUT_PREFIX_ENG,
    "k": int(K_ENG),
    "nodes": int(N),
    "valid_nodes": int(n_valid),
    "invalid_nodes": int(n_invalid),
    "parts": parts,
    "faiss": {
        "gpu": bool(gpu_used),
        "index_type": FAISS_INDEX_TYPE,
        "search_batch": int(SEARCH_BATCH_Q),
    },
    "decay": {
        "lambda_age": float(LAMBDA_AGE),
        "lambda_idle": float(LAMBDA_IDLE),
        "gamma_active": float(GAMMA_ACTIVE),
    },
    "note": "row-stochastic; cosine via inner-product on L2-normalized rows; no self-loops; only valid rows contribute"
}
man_path = TMP_DIR / f"{OUT_PREFIX_ENG}_k{K_ENG}_manifest.json"
pd.Series(man).to_json(man_path)
print(f"[MANIFEST] {man_path.name}  parts={len(parts)}, edges≈{sum(pd.read_parquet(TMP_DIR/p, engine='fastparquet').shape[0] for p in parts):,}")

print("\n[Step B3] DONE: 已生成参与度相似图 S_eng（分片 Parquet + manifest）。")


[ENG] valid rows=519,772, invalid(all-zero)=1,963
[BUILD] FAISS index for engagement ...
[READY] index built. GPU=False. Start searching...
[PROG] S_eng_topk: processed 8,192/519,772
[SAVE] part#0 edges=2,048,000 -> S_eng_topk_k50_part0000.parquet
[SAVE] part#1 edges=2,048,000 -> S_eng_topk_k50_part0001.parquet
[SAVE] part#2 edges=2,048,000 -> S_eng_topk_k50_part0002.parquet
[SAVE] part#3 edges=2,048,000 -> S_eng_topk_k50_part0003.parquet
[SAVE] part#4 edges=2,048,000 -> S_eng_topk_k50_part0004.parquet
[SAVE] part#5 edges=2,048,000 -> S_eng_topk_k50_part0005.parquet
[SAVE] part#6 edges=2,048,000 -> S_eng_topk_k50_part0006.parquet
[SAVE] part#7 edges=2,048,000 -> S_eng_topk_k50_part0007.parquet
[SAVE] part#8 edges=2,048,000 -> S_eng_topk_k50_part0008.parquet
[SAVE] part#9 edges=2,048,000 -> S_eng_topk_k50_part0009.parquet
[PROG] S_eng_topk: processed 417,792/519,772
[SAVE] part#10 edges=2,048,000 -> S_eng_topk_k50_part0010.parquet
[SAVE] part#11 edges=2,048,000 -> S_eng_topk_k50_part001

In [25]:
# =========================
# Step B4 · Params: 行为视图融合（S_ids + S_eng → S_beh）
# =========================
from pathlib import Path

TMP_DIR = Path("./tmp").resolve()

IDS_PREFIX = "S_ids_topk"     # 读取 B2 输出（row,col,val），行随机
ENG_PREFIX = "S_eng_topk"     # 读取 B3 输出（row,col,val），行随机

K_BEH              = 50       # 融合后每行保留的近邻
SAVE_PART_EDGES    = 2_000_000

# 视图全局权重（在行自适应权重的基础上再乘以这个系数，一般保持 1.0）
W_IDS_GLOBAL  = 1.0
W_ENG_GLOBAL  = 1.0

ROW_WEIGHT_EPS = 1e-12        # 防止除零

OUT_PREFIX_BEH = "S_beh_symrow"  # 输出前缀（row,col,val），行随机
IN_DOC = TMP_DIR / "doc_clean.parquet"  # 校验节点数


In [27]:
# =========================
# Step B4 · Execute
# =========================
import json
import numpy as np
import pandas as pd
from scipy import sparse
from pathlib import Path

def _require(p: Path, desc: str):
    if not p.exists():
        raise FileNotFoundError(f"[FATAL] 缺少 {desc}: {p.as_posix()}")

# 0) 预检与节点数
_require(IN_DOC, "doc_clean.parquet")
doc_df = pd.read_parquet(IN_DOC, engine="fastparquet")
N = len(doc_df)

def _load_manifest(prefix: str):
    mans = sorted(TMP_DIR.glob(f"{prefix}_k*_manifest.json"))
    if not mans:
        raise FileNotFoundError(f"[FATAL] 未找到 {prefix} 的 manifest")
    mp = mans[0]
    man = json.loads(Path(mp).read_text())
    parts = [TMP_DIR / p for p in man["parts"]]
    k = int(man.get("k", man.get("k_source", 50)))
    print(f"[MAN] {mp.name}  k={k}, parts={len(parts)}")
    return k, parts

def _build_csr(parts, n_nodes: int) -> sparse.csr_matrix:
    rows, cols, vals = [], [], []
    tot = 0
    for i, fp in enumerate(parts):
        df = pd.read_parquet(fp, engine="fastparquet")
        rows.append(df["row"].to_numpy(np.int64, copy=False))
        cols.append(df["col"].to_numpy(np.int64, copy=False))
        vals.append(df["val"].to_numpy(np.float32, copy=False))
        tot += len(df)
        if (i % 4) == 0:
            print(f"[LOAD] {i+1}/{len(parts)} parts, edges so far = {tot:,}")
        del df
    r = np.concatenate(rows) if rows else np.empty(0, dtype=np.int64)
    c = np.concatenate(cols) if cols else np.empty(0, dtype=np.int64)
    v = np.concatenate(vals) if vals else np.empty(0, dtype=np.float32)
    A = sparse.coo_matrix((v, (r, c)), shape=(n_nodes, n_nodes), dtype=np.float32).tocsr()
    A.sum_duplicates(); A.eliminate_zeros()
    print(f"[CSR] nnz={A.nnz:,}")
    return A

# 1) 读取 S_ids 与 S_eng（行随机）
print("\n[BEH] 加载 S_ids / S_eng ...")
k_ids, ids_parts = _load_manifest(IDS_PREFIX)
k_eng, eng_parts = _load_manifest(ENG_PREFIX)

P_ids = _build_csr(ids_parts, N)   # 行随机（B2 已归一）
P_eng = _build_csr(eng_parts, N)   # 行随机（B3 已归一）

# 2) 行“尖锐度” ρ = ∑ p^2
def row_rho(P: sparse.csr_matrix) -> np.ndarray:
    """
    返回每行的“尖锐度” ρ = sum_j P[i,j]^2
    使用 data^2 的前缀和计算，避免 reduceat 在空行上的越界问题。
    """
    d2 = (P.data.astype(np.float64) ** 2)
    # 前缀和：cs[k] = sum_{t < k} d2[t]，长度 = nnz + 1
    cs = np.empty(d2.size + 1, dtype=np.float64)
    cs[0] = 0.0
    np.cumsum(d2, out=cs[1:])
    ind = P.indptr
    # 每行和 = cs[ind[i+1]] - cs[ind[i]]，对空行自然为 0
    return cs[ind[1:]] - cs[ind[:-1]]


rho_ids = row_rho(P_ids)
rho_eng = row_rho(P_eng)

den = rho_ids + rho_eng + ROW_WEIGHT_EPS
alpha_ids = (rho_ids / den) * W_IDS_GLOBAL
alpha_eng = (rho_eng / den) * W_ENG_GLOBAL
den2 = alpha_ids + alpha_eng + ROW_WEIGHT_EPS
alpha_ids = alpha_ids / den2
alpha_eng = alpha_eng / den2

print("[BEH] 行自适应权重统计：")
for name, a in [("ids", alpha_ids), ("eng", alpha_eng)]:
    print(f"  - alpha_{name}: min={a.min():.4f}, median={np.median(a):.4f}, max={a.max():.4f}")

# 3) 行缩放并相加（保持稀疏）
def scale_rows(P: sparse.csr_matrix, alpha: np.ndarray) -> sparse.csr_matrix:
    P = P.tocsr(copy=True)
    deg = np.diff(P.indptr)
    rep = np.repeat(alpha, deg)
    P.data = P.data * rep.astype(np.float32)
    return P

Pt = scale_rows(P_ids, alpha_ids)
Pe = scale_rows(P_eng, alpha_eng)
P_sum = (Pt + Pe).tocsr()
P_sum.sum_duplicates(); P_sum.eliminate_zeros()
print(f"[BEH] 融合后（未裁剪） nnz={P_sum.nnz:,}")

# 4) 行内 top-K_BEH
def csr_row_topk(P: sparse.csr_matrix, k: int) -> sparse.csr_matrix:
    indptr, indices, data = P.indptr, P.indices, P.data
    N = P.shape[0]
    deg = np.diff(indptr)
    k_keep = np.minimum(k, deg)

    nnz_keep = int(k_keep.sum())
    new_indptr = np.zeros(N + 1, dtype=np.int64)
    new_indptr[1:] = np.cumsum(k_keep)

    new_indices = np.empty(nnz_keep, dtype=np.int64)
    new_data    = np.empty(nnz_keep, dtype=np.float32)

    pos = 0
    for i in range(N):
        s, e = indptr[i], indptr[i+1]
        kk = k_keep[i]
        if kk == 0: continue
        idx = indices[s:e]; val = data[s:e]
        if e - s <= k:
            new_indices[pos:pos+kk] = idx
            new_data[pos:pos+kk]    = val
        else:
            part = np.argpartition(val, -kk)[-kk:]
            ords = np.argsort(-val[part])
            sel  = part[ords]
            new_indices[pos:pos+kk] = idx[sel]
            new_data[pos:pos+kk]    = val[sel]
        pos += kk

    out = sparse.csr_matrix((new_data, new_indices, new_indptr), shape=P.shape, dtype=np.float32)
    out.eliminate_zeros()
    return out

print(f"[BEH] 行内保留 top-{K_BEH} ...")
P_topk = csr_row_topk(P_sum, K_BEH)

# 5) 行归一化（再保险）
def row_normalize(P: sparse.csr_matrix, eps: float = 1e-12) -> sparse.csr_matrix:
    rs = np.asarray(P.sum(axis=1)).ravel().astype(np.float64)
    rs[rs < eps] = 1.0
    inv = (1.0 / rs).astype(np.float32)
    Dinv = sparse.diags(inv)
    Q = Dinv @ P
    Q.eliminate_zeros()
    return Q.tocsr()

P_beh = row_normalize(P_topk)

print(f"[BEH] 最终 nnz={P_beh.nnz:,}  （≤ N*K_BEH 上限，缺边行 < K）")

# 6) 分片落盘 + manifest
def save_csr_parted(A: sparse.csr_matrix, out_prefix: str, k_val: int):
    C = A.tocoo()
    nnz = C.nnz
    print(f"[SAVE] {out_prefix}: nnz={nnz:,}")
    parts = []
    start = 0
    while start < nnz:
        end = min(start + SAVE_PART_EDGES, nnz)
        df = pd.DataFrame({
            "row": C.row[start:end].astype(np.int64),
            "col": C.col[start:end].astype(np.int64),
            "val": C.data[start:end].astype(np.float32),
        })
        outp = TMP_DIR / f"{out_prefix}_k{k_val}_part{start//SAVE_PART_EDGES:04d}.parquet"
        df.to_parquet(outp, engine="fastparquet", index=False)
        parts.append(outp.name)
        print(f"[SAVE] {outp.name} edges={len(df):,}")
        start = end
        del df
    manifest = {
        "view": out_prefix,
        "k_target": int(k_val),
        "nodes": int(A.shape[0]),
        "nnz": int(nnz),
        "parts": parts,
        "alpha_row_stat": {
            "ids": {"min": float(alpha_ids.min()), "median": float(np.median(alpha_ids)), "max": float(alpha_ids.max())},
            "eng": {"min": float(alpha_eng.min()), "median": float(np.median(alpha_eng)), "max": float(alpha_eng.max())},
        },
        "w_global": {"ids": float(W_IDS_GLOBAL), "eng": float(W_ENG_GLOBAL)},
        "note": "row-stochastic; per-row adaptive fusion of S_ids and S_eng; no self-loops"
    }
    man_path = TMP_DIR / f"{out_prefix}_k{k_val}_manifest.json"
    pd.Series(manifest).to_json(man_path)
    print(f"[MANIFEST] {man_path.name}  nnz={nnz:,}")

save_csr_parted(P_beh, OUT_PREFIX_BEH, K_BEH)
print("\n[Step B4] DONE: 已生成行为视图融合图 S_beh（分片 Parquet + manifest）。")



[BEH] 加载 S_ids / S_eng ...
[MAN] S_ids_topk_k50_manifest.json  k=50, parts=3
[MAN] S_eng_topk_k50_manifest.json  k=50, parts=13
[LOAD] 1/3 parts, edges so far = 2,535,282
[CSR] nnz=6,162,647
[LOAD] 1/13 parts, edges so far = 2,048,000
[LOAD] 5/13 parts, edges so far = 10,240,000
[LOAD] 9/13 parts, edges so far = 18,432,000
[LOAD] 13/13 parts, edges so far = 25,988,600
[CSR] nnz=25,988,600
[BEH] 行自适应权重统计：
  - alpha_ids: min=0.0000, median=0.7937, max=1.0000
  - alpha_eng: min=0.0000, median=0.2063, max=1.0000
[BEH] 融合后（未裁剪） nnz=32,126,569
[BEH] 行内保留 top-50 ...
[BEH] 最终 nnz=26,023,186  （≤ N*K_BEH 上限，缺边行 < K）
[SAVE] S_beh_symrow: nnz=26,023,186
[SAVE] S_beh_symrow_k50_part0000.parquet edges=2,000,000
[SAVE] S_beh_symrow_k50_part0001.parquet edges=2,000,000
[SAVE] S_beh_symrow_k50_part0002.parquet edges=2,000,000
[SAVE] S_beh_symrow_k50_part0003.parquet edges=2,000,000
[SAVE] S_beh_symrow_k50_part0004.parquet edges=2,000,000
[SAVE] S_beh_symrow_k50_part0005.parquet edges=2,000,000
[SAVE] 

In [28]:
# =========================
# Step C · Params: 三视图总融合（Tag + Text + Behavior）
# =========================
from pathlib import Path

TMP_DIR = Path("./tmp").resolve()

# 输入前缀（均为 Step 8 / Step B4 的“行随机”输出）
TAG_SYM_PREFIX  = "S_tag_symrow"
TEXT_SYM_PREFIX = "S_text_symrow"
BEH_SYM_PREFIX  = "S_beh_symrow"

# 融合后每行保留的近邻数
K_FUSED = 50

# 每个输出分片的最大边数
SAVE_PART_EDGES = 2_000_000

# 视图全局权重（在行自适应权重基础上再乘以全局系数；通常保持 1.0）
W_TAG_GLOBAL  = 1.0
W_TEXT_GLOBAL = 1.0
W_BEH_GLOBAL  = 1.0

ROW_WEIGHT_EPS = 1e-12

# 输出前缀（避免覆盖之前二视图结果，使用新的名称）
OUT_PREFIX_FUSED3 = "S_fused3_symrow"

# 仅用于节点数校验
IN_DOC = TMP_DIR / "doc_clean.parquet"


In [29]:
# =========================
# Step C · Execute: 三视图稀疏自适应融合
# =========================
import json
import numpy as np
import pandas as pd
from scipy import sparse
from pathlib import Path

def _require(p: Path, desc: str):
    if not p.exists():
        raise FileNotFoundError(f"[FATAL] 缺少 {desc}: {p.as_posix()}")

# 0) 节点数
_require(IN_DOC, "doc_clean.parquet")
doc_df = pd.read_parquet(IN_DOC, engine="fastparquet")
N = len(doc_df)

def _load_manifest(prefix: str):
    mans = sorted(TMP_DIR.glob(f"{prefix}_k*_manifest.json"))
    if not mans:
        raise FileNotFoundError(f"[FATAL] 未找到 {prefix} 的 manifest")
    mp = mans[0]
    man = json.loads(Path(mp).read_text())
    parts = [TMP_DIR / p for p in man["parts"]]
    k_src = int(man.get("k_target", man.get("k_source", man.get("k", 50))))
    print(f"[MAN] {mp.name}  k={k_src}, parts={len(parts)}")
    return k_src, parts

def _build_csr(parts, n_nodes: int) -> sparse.csr_matrix:
    rows, cols, vals = [], [], []
    tot = 0
    for i, fp in enumerate(parts):
        df = pd.read_parquet(fp, engine="fastparquet")
        rows.append(df["row"].to_numpy(np.int64, copy=False))
        cols.append(df["col"].to_numpy(np.int64, copy=False))
        vals.append(df["val"].to_numpy(np.float32, copy=False))
        tot += len(df)
        if (i % 5) == 0:
            print(f"[LOAD] {i+1}/{len(parts)} parts, edges so far = {tot:,}")
        del df
    if rows:
        r = np.concatenate(rows); c = np.concatenate(cols); v = np.concatenate(vals)
    else:
        r = np.empty(0, dtype=np.int64); c = np.empty(0, dtype=np.int64); v = np.empty(0, dtype=np.float32)
    A = sparse.coo_matrix((v, (r, c)), shape=(n_nodes, n_nodes), dtype=np.float32).tocsr()
    A.sum_duplicates(); A.eliminate_zeros()
    print(f"[CSR] nnz={A.nnz:,}")
    return A

def row_rho(P: sparse.csr_matrix) -> np.ndarray:
    """ρ_i = sum_j P[i,j]^2，前缀和法，兼容空行."""
    d2 = (P.data.astype(np.float64) ** 2)
    cs = np.empty(d2.size + 1, dtype=np.float64)
    cs[0] = 0.0
    np.cumsum(d2, out=cs[1:])
    ind = P.indptr
    return cs[ind[1:]] - cs[ind[:-1]]

def scale_rows(P: sparse.csr_matrix, alpha: np.ndarray) -> sparse.csr_matrix:
    P = P.tocsr(copy=True)
    deg = np.diff(P.indptr)
    rep = np.repeat(alpha, deg)
    P.data = P.data * rep.astype(np.float32)
    return P

def csr_row_topk(P: sparse.csr_matrix, k: int) -> sparse.csr_matrix:
    indptr, indices, data = P.indptr, P.indices, P.data
    N = P.shape[0]
    deg = np.diff(indptr)
    k_keep = np.minimum(k, deg)

    nnz_keep = int(k_keep.sum())
    new_indptr = np.zeros(N + 1, dtype=np.int64)
    new_indptr[1:] = np.cumsum(k_keep)

    new_indices = np.empty(nnz_keep, dtype=np.int64)
    new_data    = np.empty(nnz_keep, dtype=np.float32)

    pos = 0
    for i in range(N):
        s, e = indptr[i], indptr[i+1]
        kk = k_keep[i]
        if kk == 0: continue
        idx = indices[s:e]; val = data[s:e]
        if e - s <= k:
            new_indices[pos:pos+kk] = idx
            new_data[pos:pos+kk]    = val
        else:
            part = np.argpartition(val, -kk)[-kk:]
            ords = np.argsort(-val[part])
            sel  = part[ords]
            new_indices[pos:pos+kk] = idx[sel]
            new_data[pos:pos+kk]    = val[sel]
        pos += kk

    out = sparse.csr_matrix((new_data, new_indices, new_indptr), shape=P.shape, dtype=np.float32)
    out.eliminate_zeros()
    return out

def row_normalize(P: sparse.csr_matrix, eps: float = 1e-12) -> sparse.csr_matrix:
    rs = np.asarray(P.sum(axis=1)).ravel().astype(np.float64)
    rs[rs < eps] = 1.0
    inv = (1.0 / rs).astype(np.float32)
    Dinv = sparse.diags(inv)
    Q = Dinv @ P
    Q.eliminate_zeros()
    return Q.tocsr()

# 1) 仅为计算 α：依次加载三视图，计算 ρ 并释放
print("\n[FUSE3] 加载三视图（用于计算行自适应权重） ...")
_, tag_parts  = _load_manifest(TAG_SYM_PREFIX)
_, text_parts = _load_manifest(TEXT_SYM_PREFIX)
_, beh_parts  = _load_manifest(BEH_SYM_PREFIX)

P_tag  = _build_csr(tag_parts,  N)
rho_tag  = row_rho(P_tag);  del P_tag

P_text = _build_csr(text_parts, N)
rho_text = row_rho(P_text); del P_text

P_beh  = _build_csr(beh_parts,  N)
rho_beh  = row_rho(P_beh);  del P_beh

den = rho_tag + rho_text + rho_beh + ROW_WEIGHT_EPS
alpha_tag  = (rho_tag  / den) * W_TAG_GLOBAL
alpha_text = (rho_text / den) * W_TEXT_GLOBAL
alpha_beh  = (rho_beh  / den) * W_BEH_GLOBAL
den2 = alpha_tag + alpha_text + alpha_beh + ROW_WEIGHT_EPS
alpha_tag  = alpha_tag  / den2
alpha_text = alpha_text / den2
alpha_beh  = alpha_beh  / den2

print("[FUSE3] 行自适应权重统计：")
print(f"  - alpha_tag : min={alpha_tag.min():.4f}, median={np.median(alpha_tag):.4f}, max={alpha_tag.max():.4f}")
print(f"  - alpha_text: min={alpha_text.min():.4f}, median={np.median(alpha_text):.4f}, max={alpha_text.max():.4f}")
print(f"  - alpha_beh : min={alpha_beh.min():.4f}, median={np.median(alpha_beh):.4f}, max={alpha_beh.max():.4f}")

# 2) 逐视图加载→缩放→累加（节省内存）
print("[FUSE3] 逐视图缩放并累加 ...")
P_sum = sparse.csr_matrix((N, N), dtype=np.float32)

P_tag = _build_csr(tag_parts, N)
P_sum = (P_sum + scale_rows(P_tag,  alpha_tag)).tocsr();  del P_tag

P_text = _build_csr(text_parts, N)
P_sum = (P_sum + scale_rows(P_text, alpha_text)).tocsr(); del P_text

P_beh = _build_csr(beh_parts, N)
P_sum = (P_sum + scale_rows(P_beh,  alpha_beh)).tocsr();  del P_beh

P_sum.sum_duplicates(); P_sum.eliminate_zeros()
print(f"[FUSE3] 融合后（未裁剪） nnz={P_sum.nnz:,}")

# 3) 行内 top-K & 行归一
print(f"[FUSE3] 行内保留 top-{K_FUSED} ...")
P_topk = csr_row_topk(P_sum, K_FUSED); del P_sum
P_fused = row_normalize(P_topk); del P_topk
print(f"[FUSE3] 最终 nnz={P_fused.nnz:,} （≤ N*K_FUSED 上限）")

# 4) 分片保存 + manifest
def save_csr_parted(A: sparse.csr_matrix, out_prefix: str, k_val: int):
    C = A.tocoo()
    nnz = C.nnz
    print(f"[SAVE] {out_prefix}: nnz={nnz:,}")
    parts = []
    start = 0
    while start < nnz:
        end = min(start + SAVE_PART_EDGES, nnz)
        df = pd.DataFrame({
            "row": C.row[start:end].astype(np.int64),
            "col": C.col[start:end].astype(np.int64),
            "val": C.data[start:end].astype(np.float32),
        })
        outp = TMP_DIR / f"{out_prefix}_k{k_val}_part{start//SAVE_PART_EDGES:04d}.parquet"
        df.to_parquet(outp, engine="fastparquet", index=False)
        parts.append(outp.name)
        print(f"[SAVE] {outp.name} edges={len(df):,}")
        start = end
        del df
    man = {
        "view": out_prefix,
        "k_target": int(k_val),
        "nodes": int(A.shape[0]),
        "nnz": int(nnz),
        "parts": parts,
        "alpha_row_stat": {
            "tag":  {"min": float(alpha_tag.min()),  "median": float(np.median(alpha_tag)),  "max": float(alpha_tag.max())},
            "text": {"min": float(alpha_text.min()), "median": float(np.median(alpha_text)), "max": float(alpha_text.max())},
            "beh":  {"min": float(alpha_beh.min()),  "median": float(np.median(alpha_beh)),  "max": float(alpha_beh.max())},
        },
        "w_global": {"tag": float(W_TAG_GLOBAL), "text": float(W_TEXT_GLOBAL), "beh": float(W_BEH_GLOBAL)},
        "note": "row-stochastic; per-row adaptive fusion of tag/text/behavior; no self-loops"
    }
    man_path = TMP_DIR / f"{out_prefix}_k{k_val}_manifest.json"
    pd.Series(man).to_json(man_path)
    print(f"[MANIFEST] {man_path.name}  nnz={nnz:,}")

save_csr_parted(P_fused, OUT_PREFIX_FUSED3, K_FUSED)
print("\n[Step C] DONE: 已生成三视图融合图（分片 Parquet + manifest）。")



[FUSE3] 加载三视图（用于计算行自适应权重） ...
[MAN] S_tag_symrow_k50_manifest.json  k=50, parts=15
[MAN] S_text_symrow_k50_manifest.json  k=50, parts=15
[MAN] S_beh_symrow_k50_manifest.json  k=50, parts=14
[LOAD] 1/15 parts, edges so far = 2,000,000
[LOAD] 6/15 parts, edges so far = 12,000,000
[LOAD] 11/15 parts, edges so far = 22,000,000
[CSR] nnz=28,159,756
[LOAD] 1/15 parts, edges so far = 2,000,000
[LOAD] 6/15 parts, edges so far = 12,000,000
[LOAD] 11/15 parts, edges so far = 22,000,000
[CSR] nnz=28,161,106
[LOAD] 1/14 parts, edges so far = 2,000,000
[LOAD] 6/14 parts, edges so far = 12,000,000
[LOAD] 11/14 parts, edges so far = 22,000,000
[CSR] nnz=26,023,186
[FUSE3] 行自适应权重统计：
  - alpha_tag : min=0.0125, median=0.2021, max=0.5728
  - alpha_text: min=0.0106, median=0.2020, max=0.5796
  - alpha_beh : min=0.0000, median=0.5945, max=0.9714
[FUSE3] 逐视图缩放并累加 ...
[LOAD] 1/15 parts, edges so far = 2,000,000
[LOAD] 6/15 parts, edges so far = 12,000,000
[LOAD] 11/15 parts, edges so far = 22,000,000
[CSR]

In [30]:
# =========================
# Verify · Params
# =========================
from pathlib import Path
import numpy as np

TMP_DIR = Path("./tmp").resolve()

# 需要检查的相似网络（行随机、分片 Parquet + manifest）
VERIFY_PREFIX = "S_fused3_symrow"   # 三视图最终融合
K_EXPECT = 50                       # 期望的每行近邻数（用于统计对比）

# 数值容差（行和≈1 的容忍度）
ROW_SUM_ATOL = 1e-3

# 随机抽查邻居（显示 doc_idx 与 Kaggle Id）
NUM_SAMPLES = 3
RNG_SEED = 42


In [31]:
# =========================
# Verify · Execute
# =========================
import json, pandas as pd

def _require(p: Path, desc: str):
    assert p.exists(), f"[FATAL] 缺少 {desc}: {p.as_posix()}"

# 1) 读取 manifest 与 index_map，初始化统计缓存
man_path = sorted(TMP_DIR.glob(f"{VERIFY_PREFIX}_k*_manifest.json"))[0]
man = json.loads(man_path.read_text())
parts = [TMP_DIR / p for p in man["parts"]]
N = int(man["nodes"])
print(f"[MAN] {man_path.name}  nodes={N:,}  parts={len(parts)}  nnz={int(man['nnz']):,}")

idmap_path = TMP_DIR / "index_map.parquet"
_require(idmap_path, "index_map.parquet")
idmap = pd.read_parquet(idmap_path, engine="fastparquet").sort_values("doc_idx")
assert (idmap["doc_idx"].to_numpy() == np.arange(N)).all(), "[FATAL] index_map 未按 0..N-1 对齐"
ids = idmap["Id"].to_numpy()

row_sum = np.zeros(N, dtype=np.float64)
deg = np.zeros(N, dtype=np.int32)
val_min, val_max = np.inf, -np.inf
self_loops = 0
nan_vals = 0

# 随机抽样若干行以展示邻居（不需要记全图）
rng = np.random.default_rng(RNG_SEED)
sample_rows = rng.choice(N, size=min(NUM_SAMPLES, N), replace=False)
sample_neighbors = {int(r): [] for r in sample_rows}

# 2) 流式遍历分片，累积统计
tot_edges = 0
for i, fp in enumerate(parts, 1):
    df = pd.read_parquet(fp, engine="fastparquet")  # columns: row,col,val
    r = df["row"].to_numpy(np.int64, copy=False)
    c = df["col"].to_numpy(np.int64, copy=False)
    v = df["val"].to_numpy(np.float32, copy=False)

    # 行和 & 度数
    np.add.at(row_sum, r, v)
    np.add.at(deg, r, 1)

    # 自环/NaN/值域
    self_loops += int((r == c).sum())
    nan_vals += int(np.isnan(v).sum())
    if v.size:
        val_min = min(val_min, float(np.nanmin(v)))
        val_max = max(val_max, float(np.nanmax(v)))

    # 抽样邻居收集
    if sample_neighbors:
        mask = np.isin(r, list(sample_neighbors.keys()))
        if mask.any():
            for rr, cc, vv in zip(r[mask], c[mask], v[mask]):
                sample_neighbors[int(rr)].append((int(cc), float(vv)))

    tot_edges += len(df)
    if (i % 4) == 0 or i == len(parts):
        print(f"[LOAD] {i}/{len(parts)} parts, edges so far = {tot_edges:,}")

# 3) 统计与断言
bad_rows = int(np.sum(np.abs(row_sum - 1.0) > ROW_SUM_ATOL))
zero_rows = int(np.sum(deg == 0))
deg_min, deg_med, deg_max = int(deg.min()), float(np.median(deg)), int(deg.max())

print("\n[CHECK] Row-stochastic（行和≈1）:")
print(f"  - row_sum: min={row_sum.min():.6f}, median={np.median(row_sum):.6f}, max={row_sum.max():.6f}")
print(f"  - |row_sum-1| > {ROW_SUM_ATOL} 的行数 = {bad_rows}")

print("\n[CHECK] Degree（每行近邻数）:")
print(f"  - deg: min={deg_min}, median={deg_med:.1f}, max={deg_max}")
print(f"  - 期望 K={K_EXPECT}，等于 K 的行数 = {int(np.sum(deg == K_EXPECT)):,}")
print(f"  - 度为 0 的行数 = {zero_rows}")

print("\n[CHECK] Values（边权值域与异常）:")
print(f"  - val range: [{val_min:.6f}, {val_max:.6f}]")
print(f"  - NaN 边数: {nan_vals}")
print(f"  - 自环边数(row==col): {self_loops}")

# 4) 抽样展示邻居（按权重降序，显示 doc_idx 与 Kaggle Id）
def _preview_neighbors(r, pairs, topk=10):
    if not pairs:
        print(f"  doc_idx={r} (Id={ids[r]}): <无邻居>")
        return
    pairs = sorted(pairs, key=lambda x: -x[1])[:topk]
    txt = ", ".join([f"(doc_idx={c}, Id={ids[c]}, w={w:.4f})" for c, w in pairs])
    print(f"  doc_idx={r} (Id={ids[r]}): {txt}")

print("\n[SAMPLE] 邻居预览（按权重降序）:")
for r in sample_neighbors:
    _preview_neighbors(r, sample_neighbors[r], topk=10)

print("\n[VERIFY] Done.")


[MAN] S_fused3_symrow_k50_manifest.json  nodes=521,735  parts=14  nnz=26,086,750
[LOAD] 4/14 parts, edges so far = 8,000,000
[LOAD] 8/14 parts, edges so far = 16,000,000
[LOAD] 12/14 parts, edges so far = 24,000,000
[LOAD] 14/14 parts, edges so far = 26,086,750

[CHECK] Row-stochastic（行和≈1）:
  - row_sum: min=1.000000, median=1.000000, max=1.000000
  - |row_sum-1| > 0.001 的行数 = 0

[CHECK] Degree（每行近邻数）:
  - deg: min=50, median=50.0, max=50
  - 期望 K=50，等于 K 的行数 = 521,735
  - 度为 0 的行数 = 0

[CHECK] Values（边权值域与异常）:
  - val range: [0.000254, 0.986818]
  - NaN 边数: 0
  - 自环边数(row==col): 0

[SAMPLE] 邻居预览（按权重降序）:
  doc_idx=341512 (Id=5251400): (doc_idx=61284, Id=1011375, w=0.0228), (doc_idx=322784, Id=4969607, w=0.0222), (doc_idx=97930, Id=1513282, w=0.0207), (doc_idx=197058, Id=2956058, w=0.0205), (doc_idx=24073, Id=427174, w=0.0202), (doc_idx=26766, Id=488256, w=0.0201), (doc_idx=279599, Id=4269250, w=0.0200), (doc_idx=395842, Id=6073043, w=0.0200), (doc_idx=403939, Id=6206846, w=0.0198), (do